# E-Commerce Data Warehouse Pipeline

This notebook implements a complete data warehouse pipeline for the Udacity Data Engineering course.

**Business Context:** You are a data engineer at a fast-growing e-commerce company. Critical data is spread across multiple operational systems (PostgreSQL, Cassandra, Neo4j), making it difficult for analysts to run consistent reports. Your job is to design and implement a centralized analytics warehouse in Amazon Redshift.

## Tasks Overview

1. **Explore and Plan** - Review CSV data, identify key fields, map to warehouse schema
2. **Design the Schema** - Create dimensional model with staging, dimension, and fact tables
3. **Extract and Transform** - Load source systems and extract/transform data
4. **Load into Redshift** - Execute DDL, load staging tables, populate dimensions and facts
5. **Optimize Performance** - Apply best practices, create materialized views
6. **Validate and Report** - Run quality checks, generate final report

---
## The Data

The dataset consists of three CSV files representing data from different operational systems:

### Orders Data (PostgreSQL source) - `ecom_orders_postgres.csv`
- **order_id**: Unique identifier for each order
- **customer_id**: Customer who placed the order
- **order_datetime / ship_datetime**: Timestamps for order and shipping
- **channel / device_type / browser**: How the order was placed
- **country / state**: Geographic location
- **payment_method / campaign**: Payment and marketing info
- **Financial fields**: subtotal, discount, shipping, tax, total amounts
- **Delivery fields**: delivery_days, on_time_delivery
- **Flags**: authorization_approved, returned

### Events Data (Cassandra source) - `ecom_events_cassandra.csv`
- **event_id / session_id**: Event and session identifiers
- **customer_id**: Customer who triggered the event
- **event_type**: Type of event (page_view, product_view, add_to_cart, etc.)
- **event_ts**: Timestamp of the event
- **Device/browser/OS info**: Technical context
- **Behavioral fields**: page_depth, latency_ms, dwell_seconds
- **Commerce fields**: cart_value_usd, discount_rate, fraud_score

### Graph Edges Data (Neo4j source) - `ecom_graph_edges_neo4j.csv`
- **edge_id**: Unique relationship identifier
- **from_node_id / to_node_id**: Source and target nodes
- **from_node_type / to_node_type**: Node types (Customer, Product, Order)
- **relationship**: Type of relationship (PURCHASED, VIEWED, ADDED_TO_CART, etc.)
- **Context fields**: order_id, category, customer_segment, campaign
- **Metrics**: edge_strength, unit_price_usd, quantity

---
## Setup: Imports and Dependencies

Run this cell first to import all required libraries.

In [1]:
# ========= Imports
import os, io, re, time, json, textwrap
from datetime import datetime
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd

# Source system libraries
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from neo4j import GraphDatabase

# Warehouse (Redshift) via Data API
import boto3

# Optional progress bars
try:
    from tqdm import tqdm
    TQDM = True
except Exception:
    TQDM = False

print("All imports successful!")
print(f"   - pandas version: {pd.__version__}")
print(f"   - numpy version: {np.__version__}")

All imports successful!
   - pandas version: 2.3.1
   - numpy version: 2.2.6


---
## Setup: Configuration

Update these settings for your environment. You will need to:
1. Set your AWS credentials (from Cloud Resources)
2. Configure database connection parameters

In [2]:
# Set up AWS credentials for the session (get these from Cloud Resources)
# IMPORTANT: Replace with your actual credentials
os.environ['AWS_ACCESS_KEY_ID'] = 'YOUR_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_SECRET_ACCESS_KEY'
os.environ['AWS_SESSION_TOKEN'] = 'YOUR_SESSION_TOKEN'

In [3]:
# ========= Configuration
BASE_DIR = os.getenv("PROJECT_BASE_DIR", ".")
DATA_DIR = os.path.join(BASE_DIR, "data")
CSV_ORDERS  = os.path.join(DATA_DIR, "ecom_orders_postgres.csv")
CSV_EVENTS  = os.path.join(DATA_DIR, "ecom_events_cassandra.csv")
CSV_EDGES   = os.path.join(DATA_DIR, "ecom_graph_edges_neo4j.csv")
DDL_MD_PATH = os.path.join(BASE_DIR, "project-ddl-long.md")
MERMAID_MD  = os.path.join(BASE_DIR, "project-mermaid-diagram.md")
BATCH_SIZE  = int(os.getenv("BATCH_SIZE", "1000"))

# PostgreSQL
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB",   "postgres")
PG_USER = os.getenv("PG_USER", "temp")
PG_PW   = os.getenv("PG_PW",   "temp")

# Cassandra
CAS_HOSTS = os.getenv("CAS_HOSTS", "localhost").split(",")
CAS_PORT  = int(os.getenv("CAS_PORT", "9042"))
CAS_USER  = os.getenv("CAS_USER", "")
CAS_PW    = os.getenv("CAS_PW", "")
CAS_KEYSPACE = os.getenv("CAS_KEYSPACE", "ecommerce")

# Neo4j
NEO4J_URI  = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PW   = os.getenv("NEO4J_PW",   "neo4jpass")

# AWS/Redshift
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
REDSHIFT_DATABASE = os.getenv("REDSHIFT_DATABASE", "ecom")
REDSHIFT_WORKGROUP = os.getenv("REDSHIFT_WORKGROUP", "udacity-dwh-wg")
REDSHIFT_SECRET_ARN = os.getenv("REDSHIFT_SECRET_ARN")
REDSHIFT_CLUSTER_IDENTIFIER = os.getenv("REDSHIFT_CLUSTER_IDENTIFIER")
REDSHIFT_DB_USER = os.getenv("REDSHIFT_DB_USER")

# Verify configuration
print("Configuration loaded!")
print(f"   - BASE_DIR: {BASE_DIR}")
print(f"   - PostgreSQL: {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"   - Cassandra: {CAS_HOSTS}:{CAS_PORT}/{CAS_KEYSPACE}")
print(f"   - Neo4j: {NEO4J_URI}")
print(f"   - Redshift: {REDSHIFT_DATABASE} (workgroup: {REDSHIFT_WORKGROUP})")

print()
if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    print(f"   AWS credentials found (Key ID: {AWS_ACCESS_KEY_ID[:10]}...)")
else:
    print("   WARNING: AWS credentials NOT FOUND - set them above!")

Configuration loaded!
   - BASE_DIR: .
   - PostgreSQL: localhost:5432/postgres
   - Cassandra: ['localhost']:9042/ecommerce
   - Neo4j: bolt://localhost:7687
   - Redshift: ecom (workgroup: udacity-dwh-wg)

   AWS credentials found (Key ID: ASIA3VCJU7...)


---
## Setup: Column Specifications

These define the mapping from source columns to Redshift staging tables.
Use these as a reference when building your transformation logic.

In [4]:
# Column specs for Redshift staging (name, kind)
# kind: 's' = string, 'ts' = timestamp, 'i' = integer, 'f' = float, 'b' = boolean

ORDERS_COLSPEC = [
    ('order_id','s'),('customer_id','s'),('order_datetime','ts'),('ship_datetime','ts'),
    ('channel','s'),('device_type','s'),('browser','s'),('country','s'),('state','s'),
    ('payment_method','s'),('campaign','s'),('primary_category','s'),('num_distinct_items','i'),
    ('subtotal_usd','f'),('discount_rate','f'),('discount_amount_usd','f'),('shipping_method','s'),
    ('shipping_cost_usd','f'),('tax_rate','f'),('tax_amount_usd','f'),('order_total_usd','f'),
    ('order_weight_kg','f'),('delivery_days','i'),('on_time_delivery','b'),
    ('authorization_approved','b'),('returned','b')
]

EVENTS_COLSPEC = [
    ('event_id','s'),('customer_id','s'),('session_id','s'),('event_type','s'),('event_ts','ts'),
    ('device_type','s'),('browser','s'),('os','s'),('referrer','s'),('country','s'),('state','s'),
    ('ab_variant','s'),('is_logged_in','b'),('page_depth','i'),('latency_ms','i'),
    ('dwell_seconds','i'),('cart_value_usd','f'),('discount_rate','f'),('fraud_score','f'),
    ('payment_outcome','s'),('sequence_num','i'),('product_id','s'),('category','s'),('promo_code','s')
]

EDGES_COLSPEC = [
    ('edge_id','s'),('from_node_id','s'),('from_node_type','s'),('to_node_id','s'),('to_node_type','s'),
    ('relationship','s'),('timestamp','ts'),('order_id','s'),('category','s'),('customer_segment','s'),
    ('edge_strength','f'),('price_bucket','s'),('region','s'),('state','s'),('campaign','s'),
    ('same_household','b'),('prior_interactions','i'),('dwell_seconds','i'),('product_id','s'),
    ('unit_price_usd','f'),('quantity','i'),('returned_flag','b'),('auth_approved','b')
]

print(f"Column specs defined:")
print(f"   - Orders: {len(ORDERS_COLSPEC)} columns")
print(f"   - Events: {len(EVENTS_COLSPEC)} columns")
print(f"   - Edges: {len(EDGES_COLSPEC)} columns")

Column specs defined:
   - Orders: 26 columns
   - Events: 24 columns
   - Edges: 23 columns


---
## Setup: Helper Functions

Utility functions used throughout the pipeline.

In [5]:
def trim_df(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize text fields and handle NaN values."""
    df = df.copy()
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({'nan': np.nan, 'None': np.nan, 'NaN': np.nan, '': np.nan})
    return df

def read_csvs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Read all three source CSVs and apply cleaning."""
    orders = pd.read_csv(CSV_ORDERS)
    events = pd.read_csv(CSV_EVENTS)
    edges  = pd.read_csv(CSV_EDGES)
    return trim_df(orders), trim_df(events), trim_df(edges)

print("Helper functions defined: trim_df(), read_csvs()")

Helper functions defined: trim_df(), read_csvs()


---
# Task 1: Explore and Plan the Data Pipeline

In this task, you will:
- Review the provided CSV data files to understand structure, columns, and content
- Identify key fields and relationships important for analysis
- Map source fields to fact and dimension tables
- Consider data format standardization needs

**Deliverables:**
- Written plan mapping fields from all three sources to fact and dimension tables
- Documentation of key relationships and ID standardization strategies

In [6]:
# Read the CSV files
print("Reading CSV files...")
orders_df, events_df, edges_df = read_csvs()

print("\n" + "=" * 60)
print("TASK 1: Data Exploration")
print("=" * 60)

# Explore the orders data
print(f"\n📊 ORDERS DATA (from PostgreSQL)")
print(f"   Shape: {orders_df.shape[0]:,} rows, {orders_df.shape[1]} columns")
print(f"   Columns: {list(orders_df.columns)}")
display(orders_df.head(3))

Reading CSV files...

TASK 1: Data Exploration

📊 ORDERS DATA (from PostgreSQL)
   Shape: 2,500 rows, 26 columns
   Columns: ['order_id', 'customer_id', 'order_datetime', 'ship_datetime', 'channel', 'device_type', 'browser', 'country', 'state', 'payment_method', 'campaign', 'primary_category', 'num_distinct_items', 'subtotal_usd', 'discount_rate', 'discount_amount_usd', 'shipping_method', 'shipping_cost_usd', 'tax_rate', 'tax_amount_usd', 'order_total_usd', 'order_weight_kg', 'delivery_days', 'on_time_delivery', 'authorization_approved', 'returned']


,order_id,customer_id,order_datetime,ship_datetime,channel,device_type,browser,country,state,payment_method,...,shipping_method,shipping_cost_usd,tax_rate,tax_amount_usd,order_total_usd,order_weight_kg,delivery_days,on_time_delivery,authorization_approved,returned
0,ORD100000,C29457,2024-06-12 02:14:21,2024-06-17 02:14:21,android_app,mobile,Safari,CA,CT,google_pay,...,standard,5.05,0.0000,0.00,838.39,0.10,5,True,True,False
1,ORD100001,C22666,2024-08-08 00:16:41,2024-08-11 00:16:41,mobile_web,desktop,Opera,DE,OH,apple_pay,...,standard,9.35,0.0000,0.00,388.60,0.20,3,True,True,False
2,ORD100002,C72623,2024-09-11 06:59:18,2024-09-15 06:59:18,web,desktop,Edge,US,TX,google_pay,...,standard,6.63,0.0923,59.32,708.60,0.85,4,True,True,False


In [7]:
# Explore the events data
print(f"\n📊 EVENTS DATA (from Cassandra)")
print(f"   Shape: {events_df.shape[0]:,} rows, {events_df.shape[1]} columns")
print(f"   Columns: {list(events_df.columns)}")
display(events_df.head(3))



📊 EVENTS DATA (from Cassandra)
   Shape: 2,500 rows, 24 columns
   Columns: ['event_id', 'customer_id', 'session_id', 'event_type', 'event_ts', 'device_type', 'browser', 'os', 'referrer', 'country', 'state', 'ab_variant', 'is_logged_in', 'page_depth', 'latency_ms', 'dwell_seconds', 'cart_value_usd', 'discount_rate', 'fraud_score', 'payment_outcome', 'sequence_num', 'product_id', 'category', 'promo_code']


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200000,C47857,S7645111197,page_view,2024-11-14 00:17:09,desktop,Firefox,Linux,organic_search,IN,...,331,35,18.12,0.165,0.000,NaN,3,NaN,NaN,NONE
1,EVT200001,C83195,S1423573902,page_view,2024-08-15 10:42:18,mobile,Chrome,Linux,social,DE,...,432,26,33.62,0.083,0.262,NaN,2,NaN,NaN,NONE
2,EVT200002,C27664,S2829853742,product_view,2024-05-03 23:28:24,mobile,Chrome,Android,email,BR,...,360,6,12.26,0.237,0.162,NaN,5,P5735,Home,FREESHIP


In [8]:
# Explore the graph edges data
print(f"\n📊 GRAPH EDGES DATA (from Neo4j)")
print(f"   Shape: {edges_df.shape[0]:,} rows, {edges_df.shape[1]} columns")
print(f"   Columns: {list(edges_df.columns)}")
display(edges_df.head(3))


📊 GRAPH EDGES DATA (from Neo4j)
   Shape: 2,500 rows, 23 columns
   Columns: ['edge_id', 'from_node_id', 'from_node_type', 'to_node_id', 'to_node_type', 'relationship', 'timestamp', 'order_id', 'category', 'customer_segment', 'edge_strength', 'price_bucket', 'region', 'state', 'campaign', 'same_household', 'prior_interactions', 'dwell_seconds', 'product_id', 'unit_price_usd', 'quantity', 'returned_flag', 'auth_approved']


,edge_id,from_node_id,from_node_type,to_node_id,to_node_type,relationship,timestamp,order_id,category,customer_segment,...,state,campaign,same_household,prior_interactions,dwell_seconds,product_id,unit_price_usd,quantity,returned_flag,auth_approved
0,EDGE300000,C65482,Customer,P2261,Product,RETURNS,2024-01-27 13:42:26,ORD100938,Books,New,...,WA,SpringSale,False,3,66,P2261,315.87,2,False,False
1,EDGE300001,C59599,Customer,P7625,Product,PURCHASED,2025-02-07 10:22:55,ORD102431,Toys,Loyal,...,WI,Holiday,False,4,62,P7625,19.21,1,False,True
2,EDGE300002,C44719,Customer,P8677,Product,ADDED_TO_CART,2024-01-06 14:04:37,NaN,Pets,Active,...,KS,Loyalty,False,5,16,P8677,68.68,2,False,False


In [9]:
# Identify key fields and relationships
print("\n" + "=" * 60)
print("KEY FIELDS AND RELATIONSHIPS")
print("=" * 60)

# 1. Primary Keys (check if every row has a unique ID)
print("\n🔑 Primary Keys (Uniqueness Check):")
print(f"   - Orders (order_id):  {orders_df['order_id'].nunique():,} unique / {len(orders_df):,} rows")
print(f"   - Events (event_id):  {events_df['event_id'].nunique():,} unique / {len(events_df):,} rows")
print(f"   - Edges  (edge_id):   {edges_df['edge_id'].nunique():,} unique / {len(edges_df):,} rows")

# 2. Foreign Key Relationships (conformed entities across all 3 sources)
cust_orders = set(orders_df['customer_id'].dropna())
cust_events = set(events_df['customer_id'].dropna())
cust_edges_from = set(edges_df.loc[edges_df['from_node_type'] == 'Customer', 'from_node_id'].dropna())
cust_edges_to = set(edges_df.loc[edges_df['to_node_type'] == 'Customer', 'to_node_id'].dropna())
all_customers = cust_orders | cust_events | cust_edges_from | cust_edges_to

prod_events = set(events_df['product_id'].dropna())
prod_edges_col = set(edges_df['product_id'].dropna())
prod_edges_from = set(edges_df.loc[edges_df['from_node_type'] == 'Product', 'from_node_id'].dropna())
prod_edges_to = set(edges_df.loc[edges_df['to_node_type'] == 'Product', 'to_node_id'].dropna())
all_products = prod_events | prod_edges_col | prod_edges_from | prod_edges_to

print("\n🔗 Foreign Key Relationships & Conformed Entities:")
print(f"   - Total unique Customers across all 3 sources: {len(all_customers):,}")
print(f"     (Orders: {len(cust_orders):,}, Events: {len(cust_events):,}, Edges From: {len(cust_edges_from):,}, Edges To: {len(cust_edges_to):,})")
print(f"   - Total unique Products across Events & Edges: {len(all_products):,}")
print(f"     (Events: {len(prod_events):,}, Edges Col: {len(prod_edges_col):,}, Edges From: {len(prod_edges_from):,}, Edges To: {len(prod_edges_to):,})")
print("\n   - Graph Edge Relationship Patterns (from_node_type -> relationship -> to_node_type):")
print(edges_df.groupby(['from_node_type', 'relationship', 'to_node_type']).size().to_string())

# 3. Date/Time Fields (for dw_dim_date)
print("\n📅 Date/Time Fields (to be converted to YYYYMMDD date_key):")
print("   - Orders: order_datetime, ship_datetime")
print("   - Events: event_ts")
print("   - Edges:  timestamp")

# 4. Categorical Fields (Dimension Candidates)
print("\n📋 Categorical Fields & Cardinality (Dimension Candidates):")
cat_rows = []
for src_name, df in [('Orders', orders_df), ('Events', events_df), ('Edges', edges_df)]:
    for col in df.select_dtypes(include=['object']).columns:
        if not col.endswith(('_id', '_datetime', '_ts', 'timestamp')):
            cat_rows.append({
                'Source': src_name,
                'Column': col,
                'Unique_Count': df[col].nunique(dropna=True),
                'Null_Count': int(df[col].isna().sum()),
                'Sample_Values': ', '.join(map(str, df[col].dropna().unique()[:4]))
            })
display(pd.DataFrame(cat_rows))


KEY FIELDS AND RELATIONSHIPS

🔑 Primary Keys (Uniqueness Check):
   - Orders (order_id):  2,500 unique / 2,500 rows
   - Events (event_id):  2,500 unique / 2,500 rows
   - Edges  (edge_id):   2,500 unique / 2,500 rows

🔗 Foreign Key Relationships & Conformed Entities:
   - Total unique Customers across all 3 sources: 7,057
     (Orders: 2,460, Events: 2,461, Edges From: 2,092, Edges To: 256)
   - Total unique Products across Events & Edges: 3,360
     (Events: 1,276, Edges Col: 2,175, Edges From: 382, Edges To: 1,988)

   - Graph Edge Relationship Patterns (from_node_type -> relationship -> to_node_type):
from_node_type  relationship      to_node_type
Customer        ADDED_TO_CART     Product         441
                PURCHASED         Product         469
                REFERRED_FRIEND   Customer        256
                RETURNS           Product         111
                VIEWED            Product         830
Product         ALSO_BOUGHT_WITH  Product         393

📅 Date/Time Fi

,Source,Column,Unique_Count,Null_Count,Sample_Values
0,Orders,channel,5,0,"android_app, mobile_web, web, ios_app"
1,Orders,device_type,3,0,"mobile, desktop, tablet"
2,Orders,browser,5,0,"Safari, Opera, Edge, Chrome"
3,Orders,country,6,0,"CA, DE, US, UK"
4,Orders,state,51,0,"CT, OH, TX, VT"
5,Orders,payment_method,5,0,"google_pay, apple_pay, klarna, card"
6,Orders,campaign,6,409,"Clearance, BackToSchool, SpringSale, NewArrivals"
7,Orders,primary_category,10,0,"Books, Toys, Beauty, Sports"
8,Orders,shipping_method,3,0,"standard, two_day, overnight"
9,Events,event_type,7,0,"page_view, product_view, add_to_cart, checkout..."


In [10]:
# Data Quality Summary
print("\n" + "=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

for src_name, df in [('Orders', orders_df), ('Events', events_df), ('Edges', edges_df)]:
    null_counts = df.isna().sum()
    null_counts = null_counts[null_counts > 0]  # Keep only columns that actually have NULLs
    print(f"\n⚠️ {src_name} — Columns with NULL values:")
    if len(null_counts) == 0:
        print("   None! All columns 100% populated.")
    else:
        for col, cnt in null_counts.items():
            pct = (cnt / len(df)) * 100
            print(f"   - {col}: {cnt:,} NULLs ({pct:.1f}%)")

# Check for cross-source standardization issues (Country Code Drift)
print("\n⚠️ Cross-Source Standardization Check (Country Codes):")
print(f"   - Orders countries: {sorted(orders_df['country'].dropna().unique())}")
print(f"   - Events countries: {sorted(events_df['country'].dropna().unique())}")
print("   -> FINDING: Orders uses 'UK' while Events uses 'GB'. We will standardize 'UK' -> 'GB' in Task 3.")


DATA QUALITY SUMMARY

⚠️ Orders — Columns with NULL values:
   - campaign: 409 NULLs (16.4%)

⚠️ Events — Columns with NULL values:
   - state: 2,210 NULLs (88.4%)
   - payment_outcome: 2,127 NULLs (85.1%)
   - product_id: 1,118 NULLs (44.7%)
   - category: 1,118 NULLs (44.7%)

⚠️ Edges — Columns with NULL values:
   - order_id: 1,527 NULLs (61.1%)
   - region: 652 NULLs (26.1%)
   - campaign: 352 NULLs (14.1%)

⚠️ Cross-Source Standardization Check (Country Codes):
   - Orders countries: ['AU', 'CA', 'DE', 'FR', 'UK', 'US']
   - Events countries: ['AU', 'BR', 'CA', 'DE', 'FR', 'GB', 'IN', 'MX', 'US']
   -> FINDING: Orders uses 'UK' while Events uses 'GB'. We will standardize 'UK' -> 'GB' in Task 3.


### Task 1 Deliverable: Field Mapping Table & ID Standardization Plan

#### 1. Source-to-Target Field Mapping Table

| Source System & Field | Staging Table & Column | Target Warehouse Table & Column | Transformation / Standardizing Rule |
| :--- | :--- | :--- | :--- |
| **PG:** `order_id` | `stg_orders_raw.order_id` | `dw_fact_orders.order_id` | Trim whitespace, uppercase (`ORD...`), degenerate dimension |
| **PG:** `customer_id`<br>**CAS:** `customer_id`<br>**NEO:** `from_node_id` / `to_node_id` (`Customer`) | `stg_orders_raw.customer_id`<br>`stg_events_raw.customer_id`<br>`stg_edges_raw.from_node_id` / `to_node_id` | `dw_dim_customer.customer_id` → `customer_sk` (`from_customer_sk`, `to_customer_sk`) | Union all unique `C...` IDs across all 3 sources (7,057 total); assign `IDENTITY` surrogate key |
| **CAS:** `product_id`<br>**NEO:** `product_id`, `from_node_id` / `to_node_id` (`Product`) | `stg_events_raw.product_id`<br>`stg_edges_raw.product_id` / `from_node_id` / `to_node_id` | `dw_dim_product.product_id` → `product_sk` (`from_product_sk`, `to_product_sk`) | Union all unique `P...` IDs (3,360 total); coalesce `category`, `price_bucket`, `unit_price_usd` |
| **PG:** `order_datetime`, `ship_datetime`<br>**CAS:** `event_ts`<br>**NEO:** `timestamp` | `stg_orders_raw.order_datetime`, `ship_datetime`<br>`stg_events_raw.event_ts`<br>`stg_edges_raw.timestamp` | `dw_dim_date.date_key` (`order_date_key`, `ship_date_key`, `event_date_key`) | Extract date part and format as integer `YYYYMMDD` |
| **PG & CAS:** `country`, `state`<br>**CAS:** `is_logged_in`<br>**NEO:** `customer_segment` | `stg_*.country`, `state`, `is_logged_in`, `customer_segment` | `dw_dim_customer` attributes | Standardize country `'UK'` → `'GB'`; coalesce attributes across sources |
| **PG:** `channel`, `payment_method`, `shipping_method` | `stg_orders_raw.*` | `dw_dim_channel`, `dw_dim_payment_method`, `dw_dim_shipping_method` | Deduplicate into small lookup dimensions with surrogate keys |
| **PG & CAS:** `device_type`, `browser` | `stg_orders_raw`, `stg_events_raw` | `dw_dim_device`, `dw_dim_browser` | Conformed dimensions shared across Orders and Events |
| **PG & NEO:** `campaign` | `stg_orders_raw.campaign`, `stg_edges_raw.campaign` | `dw_dim_campaign` | Deduplicate; handle NULL campaigns gracefully |
| **CAS:** `os`, `referrer`, `ab_variant` | `stg_events_raw.*` | `dw_dim_os`, `dw_dim_referrer`, `dw_dim_ab_variant` | Deduplicate into small lookup dimensions |
| **PG:** Financial & delivery metrics (`subtotal_usd`, `order_total_usd`, `delivery_days`, flags) | `stg_orders_raw.*` | `dw_fact_orders.*` | Direct mapping at 1-row-per-order grain |
| **CAS:** Behavioral metrics (`page_depth`, `latency_ms`, `dwell_seconds`, `cart_value_usd`, `fraud_score`) | `stg_events_raw.*` | `dw_fact_events.*` | Direct mapping at 1-row-per-event grain |
| **NEO:** Graph metrics (`edge_strength`, `prior_interactions`, `dwell_seconds`, `unit_price_usd`, `quantity`) | `stg_edges_raw.*` | `dw_fact_graph_edges.*` | Split polymorphic nodes into `from/to_customer_sk` and `from/to_product_sk` |

#### 2. ID Standardization & Conforming Strategy
1. **Whitespace & Null Hygiene:** Strip whitespace from all text fields and convert stringified nulls (`'nan'`, `'None'`, `''`) into true `NULL`s.
2. **Geographic Standardization:** Convert `'UK'` in PostgreSQL Orders to `'GB'` to match ISO-2 codes in Cassandra Events.
3. **Polymorphic Graph Node Resolution:** In Neo4j, `from_node_id` and `to_node_id` can be either a `Customer` or a `Product`. In Redshift, we split these into four explicit foreign keys (`from_customer_sk`, `to_customer_sk`, `from_product_sk`, `to_product_sk`) based on `from_node_type` and `to_node_type`.
4. **Surrogate Keys:** Replace natural string keys (`C29457`, `P2261`) with integer surrogate keys (`BIGINT IDENTITY(1,1)`) in the warehouse to speed up joins and support Slowly Changing Dimensions (SCD Type 2).

---
# Task 2: Design the Warehouse Schema

In this task, you will:
- Review the dimensional (star) schema design
- Understand staging tables, dimension tables, and fact tables
- Review distribution keys, sort keys, and encoding for optimization
- Document the purpose of each table

The DDL is defined in `project-ddl-long.md`. Review the schema design and understand how it supports analytics.

**Deliverables:**
- Understanding of the provided DDL structure
- Documentation of table purposes and query support

In [11]:
# Review the DDL file
print("=" * 60)
print("TASK 2: Warehouse Schema Design")
print("=" * 60)

print("\n📄 Reading DDL file:", DDL_MD_PATH)

# 1. Read the entire Markdown DDL file into a single text string
with open(DDL_MD_PATH, 'r', encoding='utf-8') as f:
    ddl_content = f.read()

# 2. Use Regular Expressions (regex) to find every "CREATE TABLE <schema.table>"
tables = re.findall(r"CREATE\s+TABLE\s+([a-zA-Z0-9_\.]+)", ddl_content, flags=re.IGNORECASE)
views  = re.findall(r"CREATE\s+OR\s+REPLACE\s+VIEW\s+([a-zA-Z0-9_\.]+)", ddl_content, flags=re.IGNORECASE)

# 3. Categorize the tables into Staging, Dimension, and Fact lists
stg_tables  = [t for t in tables if t.startswith("stg.")]
dim_tables  = [t for t in tables if t.startswith("dw.dim_")]
fact_tables = [t for t in tables if t.startswith("dw.fact_")]

# 4. Print the inventory
print(f"\n📊 DDL Inventory Summary:")
print(f"   - Staging Tables   ({len(stg_tables)}):  {stg_tables}")
print(f"   - Dimension Tables ({len(dim_tables)}): {dim_tables}")
print(f"   - Fact Tables      ({len(fact_tables)}):  {fact_tables}")
print(f"   - Helper Views     ({len(views)}):  {views}")
print(f"   - Total Tables Defined: {len(tables)}")

TASK 2: Warehouse Schema Design

📄 Reading DDL file: ./project-ddl-long.md

📊 DDL Inventory Summary:
   - Staging Tables   (3):  ['stg.orders_raw', 'stg.events_raw', 'stg.edges_raw']
   - Dimension Tables (12): ['dw.dim_date', 'dw.dim_customer', 'dw.dim_product', 'dw.dim_campaign', 'dw.dim_channel', 'dw.dim_device', 'dw.dim_browser', 'dw.dim_os', 'dw.dim_referrer', 'dw.dim_shipping_method', 'dw.dim_payment_method', 'dw.dim_ab_variant']
   - Fact Tables      (3):  ['dw.fact_orders', 'dw.fact_events', 'dw.fact_graph_edges']
   - Helper Views     (2):  ['dw.v_lookup_customer', 'dw.v_lookup_product']
   - Total Tables Defined: 18


In [12]:
# Document the schema design
print("\n" + "=" * 60)
print("SCHEMA DESIGN DOCUMENTATION")
print("=" * 60)

schema_doc = """
## 1. Staging Tables (stg schema -> public.stg_* in Redshift)
Staging tables act as a raw, permissive landing zone (wide VARCHARs, minimal constraints)
so extracts from PostgreSQL, Cassandra, and Neo4j load reliably without failing on schema drift.
- stg.orders_raw : Raw e-commerce order transactions from PostgreSQL (26 columns)
- stg.events_raw : Raw clickstream/behavioral events from Cassandra (24 columns)
- stg.edges_raw  : Raw graph node-relationship edges from Neo4j (23 columns)

## 2. Dimension Tables (dw schema -> public.dw_dim_* in Redshift)
Dimensions store descriptive context for filtering and grouping, using BIGINT IDENTITY(1,1) surrogate keys.
- Conformed Entity Dimensions (SCD Type 2 Ready):
  * dw.dim_customer : Unified customer profile (country, state, segment, is_logged_in) + effective_from/to/is_current
  * dw.dim_product  : Unified product catalog (category, price_bucket, current_unit_price_usd) + SCD2 tracking
- Conformed Calendar Dimension:
  * dw.dim_date     : Integer smart key (YYYYMMDD) with year, quarter, month, day, week_of_year, day_of_week, is_weekend
- Small Lookup Dimensions (9 tables):
  * dw.dim_campaign, dw.dim_channel, dw.dim_device, dw.dim_browser, dw.dim_os,
    dw.dim_referrer, dw.dim_shipping_method, dw.dim_payment_method, dw.dim_ab_variant

## 3. Fact Tables & Grain Definitions (dw schema -> public.dw_fact_* in Redshift)
- dw.fact_orders      : Grain = 1 row per order (order_id). Measures revenue, discounts, shipping, tax, delivery_days, returns.
- dw.fact_events      : Grain = 1 row per clickstream event (event_id). Measures page_depth, latency_ms, dwell_seconds, cart_value, fraud_score.
- dw.fact_graph_edges : Grain = 1 row per directed graph edge (edge_id). Splits polymorphic Neo4j nodes into
                        from_customer_sk, to_customer_sk, from_product_sk, to_product_sk.

## 4. Redshift Optimization Choices
- DISTKEY(customer_sk) on fact_orders & fact_events: Collocates customer data on the same compute node for fast customer-360 joins.
- DISTKEY(to_product_sk) on fact_graph_edges: Optimizes product recommendation and co-purchase ("ALSO_BOUGHT_WITH") analytics.
- DISTSTYLE ALL on dim_date and all 9 small lookup dimensions: Broadcasts small tables to every node to eliminate network shuffling.
- SORTKEY on date keys (order_date_key, event_date_key): Enables Redshift Zone Maps to skip irrelevant blocks during time-range queries.
- ENCODE zstd: High-ratio columnar compression across text and numeric fields to minimize disk I/O.
"""

print(schema_doc)



SCHEMA DESIGN DOCUMENTATION

## 1. Staging Tables (stg schema -> public.stg_* in Redshift)
Staging tables act as a raw, permissive landing zone (wide VARCHARs, minimal constraints)
so extracts from PostgreSQL, Cassandra, and Neo4j load reliably without failing on schema drift.
- stg.orders_raw : Raw e-commerce order transactions from PostgreSQL (26 columns)
- stg.events_raw : Raw clickstream/behavioral events from Cassandra (24 columns)
- stg.edges_raw  : Raw graph node-relationship edges from Neo4j (23 columns)

## 2. Dimension Tables (dw schema -> public.dw_dim_* in Redshift)
Dimensions store descriptive context for filtering and grouping, using BIGINT IDENTITY(1,1) surrogate keys.
- Conformed Entity Dimensions (SCD Type 2 Ready):
  * dw.dim_customer : Unified customer profile (country, state, segment, is_logged_in) + effective_from/to/is_current
  * dw.dim_product  : Unified product catalog (category, price_bucket, current_unit_price_usd) + SCD2 tracking
- Conformed Calendar Dimens

### Task 2 Deliverable: Star Schema Entity-Relationship Diagram (ERD)

```mermaid
erDiagram
    dw_dim_date ||--o{ dw_fact_orders : "order_date_key / ship_date_key"
    dw_dim_date ||--o{ dw_fact_events : "event_date_key"
    dw_dim_date ||--o{ dw_fact_graph_edges : "event_date_key"

    dw_dim_customer ||--o{ dw_fact_orders : "customer_sk"
    dw_dim_customer ||--o{ dw_fact_events : "customer_sk"
    dw_dim_customer ||--o{ dw_fact_graph_edges : "from_customer_sk / to_customer_sk"

    dw_dim_product ||--o{ dw_fact_events : "product_sk"
    dw_dim_product ||--o{ dw_fact_graph_edges : "from_product_sk / to_product_sk"

    dw_dim_campaign ||--o{ dw_fact_orders : "campaign_sk"
    dw_dim_campaign ||--o{ dw_fact_graph_edges : "campaign_sk"

    dw_dim_channel ||--o{ dw_fact_orders : "channel_sk"
    dw_dim_channel ||--o{ dw_fact_events : "channel_sk"

    dw_dim_device ||--o{ dw_fact_orders : "device_sk"
    dw_dim_device ||--o{ dw_fact_events : "device_sk"

    dw_dim_browser ||--o{ dw_fact_orders : "browser_sk"
    dw_dim_browser ||--o{ dw_fact_events : "browser_sk"

    dw_dim_os ||--o{ dw_fact_events : "os_sk"
    dw_dim_referrer ||--o{ dw_fact_events : "referrer_sk"
    dw_dim_ab_variant ||--o{ dw_fact_events : "ab_variant_sk"

    dw_dim_shipmethod ||--o{ dw_fact_orders : "shipping_method_sk"
    dw_dim_paymethod  ||--o{ dw_fact_orders : "payment_method_sk"
```
*(Full column-level ERD is also maintained in `project-mermaid-diagram.md`)*

---
# Task 3: Extract and Transform the Source Data

In this task, you will:
- Load CSV data into PostgreSQL, Cassandra, and Neo4j (simulating production)
- Write extraction functions to query each source system
- Apply transformations to clean and conform the data
- Ensure transformed data matches staging table specifications

**Deliverables:**
- Working functions to connect, load, and extract from each source
- Transformed DataFrames ready for Redshift loading

## Task 3.1: Define Source System Functions

Implement the connection and data loading functions for each source system.

In [13]:
# ========= PostgreSQL Functions =========

def pg_connect():
    """Connect to PostgreSQL using psycopg2."""
    return psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PW
    )

def pg_load_orders(df: pd.DataFrame):
    """Create raw.orders table and bulk-load orders data into PostgreSQL."""
    conn = pg_connect()
    try:
        with conn.cursor() as cur:
            # 1. Create schema and reset table so re-running this cell is safe (idempotent)
            cur.execute("CREATE SCHEMA IF NOT EXISTS raw;")
            cur.execute("DROP TABLE IF EXISTS raw.orders;")
            cur.execute("""
                CREATE TABLE raw.orders (
                    order_id               VARCHAR(32) PRIMARY KEY,
                    customer_id            VARCHAR(32),
                    order_datetime         TIMESTAMP,
                    ship_datetime          TIMESTAMP,
                    channel                VARCHAR(32),
                    device_type            VARCHAR(16),
                    browser                VARCHAR(16),
                    country                VARCHAR(8),
                    state                  VARCHAR(8),
                    payment_method         VARCHAR(16),
                    campaign               VARCHAR(32),
                    primary_category       VARCHAR(32),
                    num_distinct_items     INTEGER,
                    subtotal_usd           NUMERIC(12,2),
                    discount_rate          NUMERIC(5,3),
                    discount_amount_usd    NUMERIC(12,2),
                    shipping_method        VARCHAR(16),
                    shipping_cost_usd      NUMERIC(12,2),
                    tax_rate               NUMERIC(6,4),
                    tax_amount_usd         NUMERIC(12,2),
                    order_total_usd        NUMERIC(12,2),
                    order_weight_kg        NUMERIC(10,2),
                    delivery_days          INTEGER,
                    on_time_delivery       BOOLEAN,
                    authorization_approved BOOLEAN,
                    returned               BOOLEAN
                );
            """)

            # 2. Prepare rows: convert Pandas NaN into Python None (which SQL treats as NULL)
            cols = [c for c, _ in ORDERS_COLSPEC]
            clean_df = df[cols].astype(object).where(pd.notnull(df[cols]), None)
            rows = [tuple(x) for x in clean_df.to_numpy()]

            # 3. Bulk insert all rows in one fast batch
            insert_sql = f"INSERT INTO raw.orders ({', '.join(cols)}) VALUES %s"
            execute_values(cur, insert_sql, rows, page_size=BATCH_SIZE)
        conn.commit()
        print(f"   ✓ Loaded {len(rows):,} rows into PostgreSQL (raw.orders)")
    finally:
        conn.close()

def extract_from_pg() -> pd.DataFrame:
    """Extract all orders from PostgreSQL into a Pandas DataFrame."""
    engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PW}@{PG_HOST}:{PG_PORT}/{PG_DB}")
    try:
        df = pd.read_sql_query("SELECT * FROM raw.orders;", engine)
        return df
    finally:
        engine.dispose()

print("PostgreSQL functions defined successfully!")

PostgreSQL functions defined successfully!


In [14]:
# ========= Cassandra Functions =========
from cassandra.concurrent import execute_concurrent_with_args

def cas_connect():
    """Connect to Apache Cassandra and ensure the ecommerce keyspace exists."""
    auth = PlainTextAuthProvider(CAS_USER, CAS_PW) if CAS_USER else None
    cluster = Cluster(CAS_HOSTS, port=CAS_PORT, auth_provider=auth)
    session = cluster.connect()
    # Create keyspace (Cassandra's version of a database/schema)
    session.execute(f"""
        CREATE KEYSPACE IF NOT EXISTS {CAS_KEYSPACE}
        WITH replication = {{'class': 'SimpleStrategy', 'replication_factor': 1}};
    """)
    session.set_keyspace(CAS_KEYSPACE)
    return session, cluster

def cas_load_events(df: pd.DataFrame):
    """Create events table and concurrently load events into Cassandra."""
    session, cluster = cas_connect()
    try:
        # 1. Reset table for idempotency
        session.execute("DROP TABLE IF EXISTS events;")
        session.execute("""
            CREATE TABLE events (
                event_id        text PRIMARY KEY,
                customer_id     text,
                session_id      text,
                event_type      text,
                event_ts        timestamp,
                device_type     text,
                browser         text,
                os              text,
                referrer        text,
                country         text,
                state           text,
                ab_variant      text,
                is_logged_in    boolean,
                page_depth      int,
                latency_ms      int,
                dwell_seconds   int,
                cart_value_usd  double,
                discount_rate   double,
                fraud_score     double,
                payment_outcome text,
                sequence_num    int,
                product_id      text,
                category        text,
                promo_code      text
            );
        """)

        # 2. Prepare parameterized INSERT statement
        cols = [c for c, _ in EVENTS_COLSPEC]
        placeholders = ", ".join(["?"] * len(cols))
        prepared = session.prepare(f"INSERT INTO events ({', '.join(cols)}) VALUES ({placeholders})")

        # 3. Convert event_ts to Python datetime objects and NaNs to None
        work = df[cols].copy()
        work['event_ts'] = pd.to_datetime(work['event_ts'], errors='coerce')
        rows = []
        for row in work.itertuples(index=False, name=None):
            clean_row = []
            for val in row:
                if pd.isna(val):
                    clean_row.append(None)
                elif isinstance(val, pd.Timestamp):
                    clean_row.append(val.to_pydatetime())
                else:
                    clean_row.append(val)
            rows.append(tuple(clean_row))

        # 4. Execute inserts concurrently (50 in parallel)
        execute_concurrent_with_args(session, prepared, rows, concurrency=50)
        print(f"   ✓ Loaded {len(rows):,} rows into Cassandra ({CAS_KEYSPACE}.events)")
    finally:
        cluster.shutdown()

def extract_from_cas() -> pd.DataFrame:
    """Extract all events from Cassandra into a Pandas DataFrame."""
    session, cluster = cas_connect()
    try:
        rows = session.execute("SELECT * FROM events;")
        df = pd.DataFrame(list(rows))
        return df
    finally:
        cluster.shutdown()

print("Cassandra functions defined successfully!")

Cassandra functions defined successfully!


In [15]:
# ========= Neo4j Functions =========

def neo4j_driver():
    """Create and return a Neo4j Bolt driver."""
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PW))

def neo4j_load_edges(df: pd.DataFrame):
    """Load graph edges into Neo4j as Entity nodes and INTERACTED relationships."""
    driver = neo4j_driver()
    cols = [c for c, _ in EDGES_COLSPEC]

    # Convert NaN -> None and format records as a list of dictionaries for Cypher UNWIND
    work = df[cols].copy()
    work['timestamp'] = work['timestamp'].astype(str)
    records = work.astype(object).where(pd.notnull(work), None).to_dict(orient='records')

    # Cypher query: MERGE nodes and relationships idempotently
    cypher_load = """
    UNWIND $batch AS row
    MERGE (a:Entity {id: row.from_node_id, node_type: row.from_node_type})
    MERGE (b:Entity {id: row.to_node_id,   node_type: row.to_node_type})
    MERGE (a)-[r:INTERACTED {edge_id: row.edge_id}]->(b)
    SET r.relationship       = row.relationship,
        r.timestamp          = row.timestamp,
        r.order_id           = row.order_id,
        r.category           = row.category,
        r.customer_segment   = row.customer_segment,
        r.edge_strength      = row.edge_strength,
        r.price_bucket       = row.price_bucket,
        r.region             = row.region,
        r.state              = row.state,
        r.campaign           = row.campaign,
        r.same_household     = row.same_household,
        r.prior_interactions = row.prior_interactions,
        r.dwell_seconds      = row.dwell_seconds,
        r.product_id         = row.product_id,
        r.unit_price_usd     = row.unit_price_usd,
        r.quantity           = row.quantity,
        r.returned_flag      = row.returned_flag,
        r.auth_approved      = row.auth_approved
    """

    try:
        with driver.session() as session:
            # Clear existing graph for clean idempotency
            session.run("MATCH (n) DETACH DELETE n")
            # Load in batches of 500
            batch_size = 500
            for i in range(0, len(records), batch_size):
                batch = records[i:i + batch_size]
                session.run(cypher_load, batch=batch)
        print(f"   ✓ Loaded {len(records):,} edges into Neo4j")
    finally:
        driver.close()

def extract_from_neo4j() -> pd.DataFrame:
    """Extract all relationships from Neo4j back into a tabular Pandas DataFrame."""
    driver = neo4j_driver()
    cypher_extract = """
    MATCH (a:Entity)-[r:INTERACTED]->(b:Entity)
    RETURN
        r.edge_id            AS edge_id,
        a.id                 AS from_node_id,
        a.node_type          AS from_node_type,
        b.id                 AS to_node_id,
        b.node_type          AS to_node_type,
        r.relationship       AS relationship,
        r.timestamp          AS timestamp,
        r.order_id           AS order_id,
        r.category           AS category,
        r.customer_segment   AS customer_segment,
        r.edge_strength      AS edge_strength,
        r.price_bucket       AS price_bucket,
        r.region             AS region,
        r.state              AS state,
        r.campaign           AS campaign,
        r.same_household     AS same_household,
        r.prior_interactions AS prior_interactions,
        r.dwell_seconds      AS dwell_seconds,
        r.product_id         AS product_id,
        r.unit_price_usd     AS unit_price_usd,
        r.quantity           AS quantity,
        r.returned_flag      AS returned_flag,
        r.auth_approved      AS auth_approved
    """
    try:
        with driver.session() as session:
            result = session.run(cypher_extract)
            df = pd.DataFrame([record.data() for record in result])
        return df
    finally:
        driver.close()

print("Neo4j functions defined successfully!")

Neo4j functions defined successfully!


## Task 3.2: Load Source Systems

Load the CSV data into the operational databases (simulating production environment).

In [16]:
print("=" * 60)
print("TASK 3: Loading Source Systems")
print("=" * 60)

# Load PostgreSQL
print("\n📦 Loading orders into PostgreSQL...")
pg_load_orders(orders_df)

# Load Cassandra
print("\n📦 Loading events into Cassandra...")
cas_load_events(events_df)

# Load Neo4j
print("\n📦 Loading edges into Neo4j...")
neo4j_load_edges(edges_df)

print("\n✅ All source systems loaded!")

TASK 3: Loading Source Systems

📦 Loading orders into PostgreSQL...
   ✓ Loaded 2,500 rows into PostgreSQL (raw.orders)

📦 Loading events into Cassandra...
   ✓ Loaded 2,500 rows into Cassandra (ecommerce.events)

📦 Loading edges into Neo4j...
   ✓ Loaded 2,500 edges into Neo4j

✅ All source systems loaded!


## Task 3.3: Extract and Transform

Extract data from source systems and transform for Redshift staging.

In [17]:
print("\n" + "=" * 60)
print("Extracting from Source Systems")
print("=" * 60)

# 1. Extract from all 3 live operational databases
print("\n📤 Extracting from PostgreSQL...")
orders_extracted = extract_from_pg()
print(f"   Extracted {len(orders_extracted):,} orders")

print("\n📤 Extracting from Cassandra...")
events_extracted = extract_from_cas()
print(f"   Extracted {len(events_extracted):,} events")

print("\n📤 Extracting from Neo4j...")
edges_extracted = extract_from_neo4j()
print(f"   Extracted {len(edges_extracted):,} edges")

# 2. Apply Transformations & Print Before/After Proof
print("\n" + "=" * 60)
print("Applying Transformations (Before vs. After)")
print("=" * 60)

# Transformation A: Standardize Country Code ('UK' -> 'GB') in Orders
uk_before = (orders_extracted['country'] == 'UK').sum()
orders_extracted['country'] = orders_extracted['country'].replace({'UK': 'GB'})
uk_after = (orders_extracted['country'] == 'UK').sum()
gb_after = (orders_extracted['country'] == 'GB').sum()
print(f"\n🔄 1. Country Standardization (Orders):")
print(f"   - BEFORE: 'UK' count = {uk_before:,} | Countries: {sorted(orders_df['country'].dropna().unique())}")
print(f"   - AFTER:  'UK' count = {uk_after:,}, 'GB' count = {gb_after:,} | Countries: {sorted(orders_extracted['country'].dropna().unique())}")

# Transformation B: Conform IDs (Trim + Uppercase) & Clean Categories
for df_temp, id_cols in [
    (orders_extracted, ['order_id', 'customer_id']),
    (events_extracted, ['event_id', 'customer_id', 'product_id']),
    (edges_extracted,  ['edge_id', 'from_node_id', 'to_node_id', 'order_id', 'product_id'])
]:
    for col in id_cols:
        if col in df_temp.columns:
            df_temp[col] = df_temp[col].apply(lambda x: str(x).strip().upper() if pd.notnull(x) else None)

# Fill missing campaigns with 'Unknown' for clean dimension joins
orders_null_camp_before = orders_extracted['campaign'].isna().sum()
orders_extracted['campaign'] = orders_extracted['campaign'].fillna('Unknown')
edges_extracted['campaign']  = edges_extracted['campaign'].fillna('Unknown')
print(f"\n🔄 2. Category Cleaning (Campaign NULLs -> 'Unknown'):")
print(f"   - Orders campaign NULLs BEFORE: {orders_null_camp_before:,} -> AFTER: {orders_extracted['campaign'].isna().sum():,}")

# Transformation C: Derived Date Keys Preview (YYYYMMDD)
orders_date_preview = pd.DataFrame({
    'order_id': orders_extracted['order_id'].head(3),
    'order_datetime_raw': orders_extracted['order_datetime'].head(3),
    'derived_order_date_key': pd.to_datetime(orders_extracted['order_datetime'].head(3)).dt.strftime('%Y%m%d').astype(int)
})
print("\n🔄 3. Derived Date Key Verification (Sample):")
display(orders_date_preview)

# 3. Conform final DataFrames to exact Redshift Staging COLSPEC order
print("\n🔄 Conforming DataFrames to Redshift Staging Column Specs...")
orders = orders_extracted[[c for c, _ in ORDERS_COLSPEC]].copy()
events = events_extracted[[c for c, _ in EVENTS_COLSPEC]].copy()
edges  = edges_extracted[[c for c, _ in EDGES_COLSPEC]].copy()

print(f"   - orders aligned: {orders.shape} (matches {len(ORDERS_COLSPEC)} COLSPEC cols)")
print(f"   - events aligned: {events.shape} (matches {len(EVENTS_COLSPEC)} COLSPEC cols)")
print(f"   - edges  aligned: {edges.shape} (matches {len(EDGES_COLSPEC)} COLSPEC cols)")

print("\n✅ Task 3 Complete - Data extracted, transformed, and conformed!")


Extracting from Source Systems

📤 Extracting from PostgreSQL...
   Extracted 2,500 orders

📤 Extracting from Cassandra...
   Extracted 2,500 events

📤 Extracting from Neo4j...
   Extracted 2,500 edges

Applying Transformations (Before vs. After)

🔄 1. Country Standardization (Orders):
   - BEFORE: 'UK' count = 66 | Countries: ['AU', 'CA', 'DE', 'FR', 'UK', 'US']
   - AFTER:  'UK' count = 0, 'GB' count = 66 | Countries: ['AU', 'CA', 'DE', 'FR', 'GB', 'US']

🔄 2. Category Cleaning (Campaign NULLs -> 'Unknown'):
   - Orders campaign NULLs BEFORE: 409 -> AFTER: 0

🔄 3. Derived Date Key Verification (Sample):


,order_id,order_datetime_raw,derived_order_date_key
0,ORD100000,2024-06-12 02:14:21,20240612
1,ORD100001,2024-08-08 00:16:41,20240808
2,ORD100002,2024-09-11 06:59:18,20240911



🔄 Conforming DataFrames to Redshift Staging Column Specs...
   - orders aligned: (2500, 26) (matches 26 COLSPEC cols)
   - events aligned: (2500, 24) (matches 24 COLSPEC cols)
   - edges  aligned: (2500, 23) (matches 23 COLSPEC cols)

✅ Task 3 Complete - Data extracted, transformed, and conformed!


---
# Task 4: Load Data into Redshift

In this task, you will:
- Execute the DDL to create staging, dimension, and fact tables
- Load data into Redshift staging tables
- Populate dimension tables from staging data
- Populate fact tables with dimension key lookups
- Validate successful loading

**Deliverables:**
- Working Redshift connection and execution functions
- Loaded staging, dimension, and fact tables
- Row count validation

## Task 4.1: Define Redshift Functions

In [18]:
# ========= Redshift Functions =========

session_boto = boto3.Session(region_name=AWS_REGION)
rsd = session_boto.client("redshift-data", region_name=AWS_REGION)

def _rs_kwargs() -> Dict[str, Any]:
    """Build Redshift Data API connection parameters."""
    base = dict(Database=REDSHIFT_DATABASE)
    if REDSHIFT_WORKGROUP:
        base["WorkgroupName"] = REDSHIFT_WORKGROUP
        if REDSHIFT_SECRET_ARN:
            base["SecretArn"] = REDSHIFT_SECRET_ARN
    elif REDSHIFT_CLUSTER_IDENTIFIER and REDSHIFT_DB_USER:
        base["ClusterIdentifier"] = REDSHIFT_CLUSTER_IDENTIFIER
        base["DbUser"] = REDSHIFT_DB_USER
    else:
        raise RuntimeError("Configure Redshift serverless OR provisioned for Data API.")
    return base

def rs_exec(sql: str, return_results: bool = False, timeout_s: int = 900):
    """Execute a SQL statement on Amazon Redshift via the Data API and poll until complete."""
    # 1. Submit SQL asynchronously to Redshift
    resp = rsd.execute_statement(Sql=sql, **_rs_kwargs())
    stmt_id = resp["Id"]

    # 2. Poll until FINISHED, FAILED, or ABORTED
    start = time.time()
    while True:
        desc = rsd.describe_statement(Id=stmt_id)
        status = desc["Status"]
        if status == "FINISHED":
            break
        if status in ("FAILED", "ABORTED"):
            raise RuntimeError(f"Redshift SQL {status}: {desc.get('Error')}\nSQL snippet:\n{sql[:400]}")
        if time.time() - start > timeout_s:
            raise TimeoutError(f"Redshift statement {stmt_id} timed out after {timeout_s}s")
        time.sleep(0.5)

    # 3. If caller wants rows back (SELECT queries), fetch and convert to list of dicts
    if not return_results:
        return None

    rows_out = []
    next_token = None
    while True:
        kwargs = {"Id": stmt_id}
        if next_token:
            kwargs["NextToken"] = next_token
        res = rsd.get_statement_result(**kwargs)
        col_names = [c["name"] for c in res["ColumnMetadata"]]
        for record in res["Records"]:
            row_dict = {}
            for col_name, cell in zip(col_names, record):
                if cell.get("isNull"):
                    row_dict[col_name] = None
                elif "stringValue" in cell:
                    row_dict[col_name] = cell["stringValue"]
                elif "longValue" in cell:
                    row_dict[col_name] = cell["longValue"]
                elif "doubleValue" in cell:
                    row_dict[col_name] = cell["doubleValue"]
                elif "booleanValue" in cell:
                    row_dict[col_name] = cell["booleanValue"]
                else:
                    row_dict[col_name] = None
            rows_out.append(row_dict)
        next_token = res.get("NextToken")
        if not next_token:
            break
    return rows_out

def _format_sql_val(val, kind: str) -> str:
    """Format a single Python/Pandas value into a safe Redshift SQL literal."""
    if pd.isna(val) or val is None:
        return "NULL"
    if kind in ("s", "ts"):
        s = str(val).replace("'", "''")  # Escape single quotes for SQL safety
        return f"'{s}'"
    if kind == "i":
        return str(int(val))
    if kind == "f":
        return str(float(val))
    if kind == "b":
        return "TRUE" if bool(val) else "FALSE"
    return f"'{str(val)}'"

def rs_batch_insert(table: str, colspec: List[Tuple[str, str]], df: pd.DataFrame, chunk_size: int = 200):
    """Load a DataFrame into Redshift using chunked multi-row INSERT statements."""
    cols = [c for c, _ in colspec]
    col_list_sql = ", ".join(cols)
    total_rows = len(df)
    inserted = 0

    for start in range(0, total_rows, chunk_size):
        chunk = df.iloc[start:start + chunk_size]
        value_tuples = []
        for row in chunk[cols].itertuples(index=False, name=None):
            formatted = [_format_sql_val(v, kind) for v, (_, kind) in zip(row, colspec)]
            value_tuples.append(f"({', '.join(formatted)})")
        sql = f"INSERT INTO {table} ({col_list_sql}) VALUES\n" + ",\n".join(value_tuples) + ";"
        rs_exec(sql)
        inserted += len(chunk)
    print(f"   ✓ Inserted {inserted:,} rows into {table}")

print("Redshift Data API functions defined successfully!")

Redshift Data API functions defined successfully!


## Task 4.2: Execute DDL and Create Tables

In [19]:
print("=" * 60)
print("TASK 4: Loading Data into Redshift")
print("=" * 60)

print("\n📋 Step 1: Executing DDL to create staging, dimension, and fact tables...")

# 1. Read project-ddl-long.md and extract all ```sql ... ``` blocks
with open(DDL_MD_PATH, "r", encoding="utf-8") as f:
    md_content = f.read()

sql_blocks = re.findall(r"```sql(.*?)```", md_content, flags=re.DOTALL | re.IGNORECASE)

# 2. Split into individual statements, rewrite schemas to public.*, and execute
executed_count = 0
for block in sql_blocks:
    for raw_stmt in block.split(";"):
        # Remove SQL single-line comments (-- ...) and blank whitespace
        lines = [ln for ln in raw_stmt.splitlines() if not ln.strip().startswith("--")]
        stmt = "\n".join(lines).strip()
        if not stmt:
            continue
        # Skip CREATE SCHEMA (we use the default public schema)
        if stmt.upper().startswith("CREATE SCHEMA"):
            continue
        # Rewrite stg.table -> public.stg_table and dw.table -> public.dw_table
        stmt = re.sub(r"\bstg\.", "public.stg_", stmt)
        stmt = re.sub(r"\bdw\.", "public.dw_", stmt)
        # Add CASCADE to DROP TABLE so dependent views/MVs don't block re-runs
        if stmt.upper().startswith("DROP TABLE") and "CASCADE" not in stmt.upper():
            stmt += " CASCADE"
        rs_exec(stmt + ";")
        executed_count += 1

print(f"   ✓ Successfully executed {executed_count} DDL statements in Redshift!")

TASK 4: Loading Data into Redshift

📋 Step 1: Executing DDL to create staging, dimension, and fact tables...
   ✓ Successfully executed 38 DDL statements in Redshift!


## Task 4.3: Load Staging Tables

In [20]:
print("\n📦 Step 2: Loading staging tables...")

print("\n  Loading orders into public.stg_orders_raw...")
rs_exec("TRUNCATE TABLE public.stg_orders_raw;")
rs_batch_insert("public.stg_orders_raw", ORDERS_COLSPEC, orders)

print("\n  Loading events into public.stg_events_raw...")
rs_exec("TRUNCATE TABLE public.stg_events_raw;")
rs_batch_insert("public.stg_events_raw", EVENTS_COLSPEC, events)

print("\n  Loading edges into public.stg_edges_raw...")
rs_exec("TRUNCATE TABLE public.stg_edges_raw;")
rs_batch_insert("public.stg_edges_raw", EDGES_COLSPEC, edges)

print("\n✅ Staging tables loaded!")


📦 Step 2: Loading staging tables...

  Loading orders into public.stg_orders_raw...
   ✓ Inserted 2,500 rows into public.stg_orders_raw

  Loading events into public.stg_events_raw...
   ✓ Inserted 2,500 rows into public.stg_events_raw

  Loading edges into public.stg_edges_raw...
   ✓ Inserted 2,500 rows into public.stg_edges_raw

✅ Staging tables loaded!


## Task 4.4: Populate Dimension Tables

In [21]:
print("\n📊 Step 3: Populating all 12 dimension tables...")

# 1. Date Dimension (conformed across Orders, Events, and Graph Edges)
rs_exec("TRUNCATE TABLE public.dw_dim_date;")
rs_exec("""
    INSERT INTO public.dw_dim_date (
        date_key, date_actual, year, quarter, month, day, week_of_year, day_of_week, is_weekend
    )
    SELECT DISTINCT
        CAST(TO_CHAR(dt, 'YYYYMMDD') AS INTEGER) AS date_key,
        dt AS date_actual,
        EXTRACT(YEAR FROM dt)::SMALLINT,
        EXTRACT(QUARTER FROM dt)::SMALLINT,
        EXTRACT(MONTH FROM dt)::SMALLINT,
        EXTRACT(DAY FROM dt)::SMALLINT,
        EXTRACT(WEEK FROM dt)::SMALLINT,
        EXTRACT(DOW FROM dt)::SMALLINT,
        (EXTRACT(DOW FROM dt) IN (0, 6))::BOOLEAN
    FROM (
        SELECT order_datetime::DATE AS dt FROM public.stg_orders_raw WHERE order_datetime IS NOT NULL
        UNION
        SELECT ship_datetime::DATE  AS dt FROM public.stg_orders_raw WHERE ship_datetime IS NOT NULL
        UNION
        SELECT event_ts::DATE       AS dt FROM public.stg_events_raw WHERE event_ts IS NOT NULL
        UNION
        SELECT timestamp::DATE      AS dt FROM public.stg_edges_raw  WHERE timestamp IS NOT NULL
    ) dates
    WHERE dt IS NOT NULL;
""")
print("   ✓ Populated dw_dim_date")

# 2. Customer Dimension (Conformed across all 3 sources, SCD Type 2 ready)
rs_exec("TRUNCATE TABLE public.dw_dim_customer;")
rs_exec("""
    INSERT INTO public.dw_dim_customer (
        customer_id, country, state, customer_segment, is_logged_in,
        effective_from, effective_to, is_current
    )
    WITH all_cust AS (
        SELECT customer_id, country, state, NULL::VARCHAR(16) AS customer_segment, NULL::BOOLEAN AS is_logged_in
        FROM public.stg_orders_raw WHERE customer_id IS NOT NULL
        UNION ALL
        SELECT customer_id, country, state, NULL::VARCHAR(16), is_logged_in
        FROM public.stg_events_raw WHERE customer_id IS NOT NULL
        UNION ALL
        SELECT from_node_id AS customer_id, NULL::VARCHAR(8), state, customer_segment, NULL::BOOLEAN
        FROM public.stg_edges_raw WHERE from_node_type = 'Customer' AND from_node_id IS NOT NULL
        UNION ALL
        SELECT to_node_id AS customer_id, NULL::VARCHAR(8), NULL::VARCHAR(8), NULL::VARCHAR(16), NULL::BOOLEAN
        FROM public.stg_edges_raw WHERE to_node_type = 'Customer' AND to_node_id IS NOT NULL
    )
    SELECT
        customer_id,
        MAX(country)          AS country,
        MAX(state)            AS state,
        MAX(customer_segment) AS customer_segment,
        BOOL_OR(is_logged_in) AS is_logged_in,
        CURRENT_TIMESTAMP     AS effective_from,
        NULL::TIMESTAMP       AS effective_to,
        TRUE                  AS is_current
    FROM all_cust
    GROUP BY customer_id;
""")
print("   ✓ Populated dw_dim_customer")

# 3. Product Dimension (Conformed across Events and Graph Edges, SCD Type 2 ready)
rs_exec("TRUNCATE TABLE public.dw_dim_product;")
rs_exec("""
    INSERT INTO public.dw_dim_product (
        product_id, category, price_bucket, current_unit_price_usd,
        effective_from, effective_to, is_current
    )
    WITH all_prod AS (
        SELECT product_id, category, NULL::VARCHAR(16) AS price_bucket, NULL::DECIMAL(12,2) AS unit_price_usd
        FROM public.stg_events_raw WHERE product_id IS NOT NULL
        UNION ALL
        SELECT product_id, category, price_bucket, unit_price_usd
        FROM public.stg_edges_raw WHERE product_id IS NOT NULL
        UNION ALL
        SELECT from_node_id AS product_id, category, price_bucket, unit_price_usd
        FROM public.stg_edges_raw WHERE from_node_type = 'Product' AND from_node_id IS NOT NULL
        UNION ALL
        SELECT to_node_id AS product_id, category, price_bucket, unit_price_usd
        FROM public.stg_edges_raw WHERE to_node_type = 'Product' AND to_node_id IS NOT NULL
    )
    SELECT
        product_id,
        MAX(category)       AS category,
        MAX(price_bucket)   AS price_bucket,
        MAX(unit_price_usd) AS current_unit_price_usd,
        CURRENT_TIMESTAMP   AS effective_from,
        NULL::TIMESTAMP     AS effective_to,
        TRUE                AS is_current
    FROM all_prod
    GROUP BY product_id;
""")
print("   ✓ Populated dw_dim_product")

# 4. Small Lookup Dimensions (9 tables)
lookup_sqls = {
    "dw_dim_campaign": """
        INSERT INTO public.dw_dim_campaign (campaign)
        SELECT DISTINCT campaign FROM (
            SELECT campaign FROM public.stg_orders_raw WHERE campaign IS NOT NULL
            UNION
            SELECT campaign FROM public.stg_edges_raw  WHERE campaign IS NOT NULL
        );
    """,
    "dw_dim_channel": """
        INSERT INTO public.dw_dim_channel (channel)
        SELECT DISTINCT channel FROM public.stg_orders_raw WHERE channel IS NOT NULL;
    """,
    "dw_dim_device": """
        INSERT INTO public.dw_dim_device (device_type)
        SELECT DISTINCT device_type FROM (
            SELECT device_type FROM public.stg_orders_raw WHERE device_type IS NOT NULL
            UNION
            SELECT device_type FROM public.stg_events_raw WHERE device_type IS NOT NULL
        );
    """,
    "dw_dim_browser": """
        INSERT INTO public.dw_dim_browser (browser)
        SELECT DISTINCT browser FROM (
            SELECT browser FROM public.stg_orders_raw WHERE browser IS NOT NULL
            UNION
            SELECT browser FROM public.stg_events_raw WHERE browser IS NOT NULL
        );
    """,
    "dw_dim_os": """
        INSERT INTO public.dw_dim_os (os)
        SELECT DISTINCT os FROM public.stg_events_raw WHERE os IS NOT NULL;
    """,
    "dw_dim_referrer": """
        INSERT INTO public.dw_dim_referrer (referrer)
        SELECT DISTINCT referrer FROM public.stg_events_raw WHERE referrer IS NOT NULL;
    """,
    "dw_dim_shipping_method": """
        INSERT INTO public.dw_dim_shipping_method (shipping_method)
        SELECT DISTINCT shipping_method FROM public.stg_orders_raw WHERE shipping_method IS NOT NULL;
    """,
    "dw_dim_payment_method": """
        INSERT INTO public.dw_dim_payment_method (payment_method)
        SELECT DISTINCT payment_method FROM public.stg_orders_raw WHERE payment_method IS NOT NULL;
    """,
    "dw_dim_ab_variant": """
        INSERT INTO public.dw_dim_ab_variant (ab_variant)
        SELECT DISTINCT ab_variant FROM public.stg_events_raw WHERE ab_variant IS NOT NULL;
    """
}

for dim_name, sql in lookup_sqls.items():
    rs_exec(f"TRUNCATE TABLE public.{dim_name};")
    rs_exec(sql)
    print(f"   ✓ Populated {dim_name}")

print("\n✅ All 12 dimension tables populated!")


📊 Step 3: Populating all 12 dimension tables...
   ✓ Populated dw_dim_date
   ✓ Populated dw_dim_customer
   ✓ Populated dw_dim_product
   ✓ Populated dw_dim_campaign
   ✓ Populated dw_dim_channel
   ✓ Populated dw_dim_device
   ✓ Populated dw_dim_browser
   ✓ Populated dw_dim_os
   ✓ Populated dw_dim_referrer
   ✓ Populated dw_dim_shipping_method
   ✓ Populated dw_dim_payment_method
   ✓ Populated dw_dim_ab_variant

✅ All 12 dimension tables populated!


## Task 4.5: Populate Fact Tables

In [22]:
print("\n📊 Step 4: Populating all 3 fact tables...")

# 1. Fact: Orders
rs_exec("TRUNCATE TABLE public.dw_fact_orders;")
rs_exec("""
    INSERT INTO public.dw_fact_orders (
        order_id, customer_sk, order_date_key, ship_date_key,
        channel_sk, device_sk, browser_sk, campaign_sk,
        payment_method_sk, shipping_method_sk, primary_category,
        num_distinct_items, subtotal_usd, discount_rate, discount_amount_usd,
        shipping_cost_usd, tax_rate, tax_amount_usd, order_total_usd,
        order_weight_kg, delivery_days, on_time_delivery,
        authorization_approved, returned
    )
    SELECT
        o.order_id,
        dc.customer_sk,
        CAST(TO_CHAR(o.order_datetime::DATE, 'YYYYMMDD') AS INTEGER),
        CAST(TO_CHAR(o.ship_datetime::DATE,  'YYYYMMDD') AS INTEGER),
        ch.channel_sk,
        dv.device_sk,
        br.browser_sk,
        cp.campaign_sk,
        pm.payment_method_sk,
        sm.shipping_method_sk,
        o.primary_category,
        o.num_distinct_items, o.subtotal_usd, o.discount_rate, o.discount_amount_usd,
        o.shipping_cost_usd, o.tax_rate, o.tax_amount_usd, o.order_total_usd,
        o.order_weight_kg, o.delivery_days, o.on_time_delivery,
        o.authorization_approved, o.returned
    FROM public.stg_orders_raw o
    LEFT JOIN public.dw_dim_customer        dc ON dc.customer_id = o.customer_id AND dc.is_current = TRUE
    LEFT JOIN public.dw_dim_channel         ch ON ch.channel = o.channel
    LEFT JOIN public.dw_dim_device          dv ON dv.device_type = o.device_type
    LEFT JOIN public.dw_dim_browser         br ON br.browser = o.browser
    LEFT JOIN public.dw_dim_campaign        cp ON cp.campaign = o.campaign
    LEFT JOIN public.dw_dim_payment_method  pm ON pm.payment_method = o.payment_method
    LEFT JOIN public.dw_dim_shipping_method sm ON sm.shipping_method = o.shipping_method;
""")
print("   ✓ Populated dw_fact_orders")

# 2. Fact: Events
rs_exec("TRUNCATE TABLE public.dw_fact_events;")
rs_exec("""
    INSERT INTO public.dw_fact_events (
        event_id, customer_sk, product_sk, event_date_key, session_id,
        event_type, channel_sk, device_sk, browser_sk, os_sk, referrer_sk, ab_variant_sk,
        page_depth, latency_ms, dwell_seconds, cart_value_usd,
        discount_rate, fraud_score, payment_outcome, sequence_num, category, promo_code
    )
    SELECT
        e.event_id,
        dc.customer_sk,
        dp.product_sk,
        CAST(TO_CHAR(e.event_ts::DATE, 'YYYYMMDD') AS INTEGER),
        e.session_id,
        e.event_type,
        NULL::BIGINT AS channel_sk,
        dv.device_sk,
        br.browser_sk,
        os.os_sk,
        rf.referrer_sk,
        ab.ab_variant_sk,
        e.page_depth, e.latency_ms, e.dwell_seconds, e.cart_value_usd,
        e.discount_rate, e.fraud_score, e.payment_outcome, e.sequence_num, e.category, e.promo_code
    FROM public.stg_events_raw e
    LEFT JOIN public.dw_dim_customer   dc ON dc.customer_id = e.customer_id AND dc.is_current = TRUE
    LEFT JOIN public.dw_dim_product    dp ON dp.product_id = e.product_id   AND dp.is_current = TRUE
    LEFT JOIN public.dw_dim_device     dv ON dv.device_type = e.device_type
    LEFT JOIN public.dw_dim_browser    br ON br.browser = e.browser
    LEFT JOIN public.dw_dim_os         os ON os.os = e.os
    LEFT JOIN public.dw_dim_referrer   rf ON rf.referrer = e.referrer
    LEFT JOIN public.dw_dim_ab_variant ab ON ab.ab_variant = e.ab_variant;
""")
print("   ✓ Populated dw_fact_events")

# 3. Fact: Graph Edges (Polymorphic node resolution into 4 explicit role-playing FKs)
rs_exec("TRUNCATE TABLE public.dw_fact_graph_edges;")
rs_exec("""
    INSERT INTO public.dw_fact_graph_edges (
        edge_id, event_date_key, relationship,
        from_customer_sk, to_customer_sk, from_product_sk, to_product_sk,
        order_id, category, campaign_sk, customer_segment, region, state,
        edge_strength, price_bucket, prior_interactions, dwell_seconds,
        unit_price_usd, quantity, returned_flag, auth_approved
    )
    SELECT
        g.edge_id,
        CAST(TO_CHAR(g.timestamp::DATE, 'YYYYMMDD') AS INTEGER),
        g.relationship,
        fc.customer_sk AS from_customer_sk,
        tc.customer_sk AS to_customer_sk,
        fp.product_sk  AS from_product_sk,
        tp.product_sk  AS to_product_sk,
        g.order_id, g.category,
        cp.campaign_sk,
        g.customer_segment, g.region, g.state,
        g.edge_strength, g.price_bucket, g.prior_interactions, g.dwell_seconds,
        g.unit_price_usd, g.quantity, g.returned_flag, g.auth_approved
    FROM public.stg_edges_raw g
    LEFT JOIN public.dw_dim_customer fc ON g.from_node_type = 'Customer' AND fc.customer_id = g.from_node_id AND fc.is_current = TRUE
    LEFT JOIN public.dw_dim_customer tc ON g.to_node_type   = 'Customer' AND tc.customer_id = g.to_node_id   AND tc.is_current = TRUE
    LEFT JOIN public.dw_dim_product  fp ON g.from_node_type = 'Product'  AND fp.product_id  = g.from_node_id AND fp.is_current = TRUE
    LEFT JOIN public.dw_dim_product  tp ON g.to_node_type   = 'Product'  AND tp.product_id  = g.to_node_id   AND tp.is_current = TRUE
    LEFT JOIN public.dw_dim_campaign cp ON cp.campaign = g.campaign;
""")
print("   ✓ Populated dw_fact_graph_edges")

# 4. Completeness Validation: Prove Staging == Final Fact Row Counts (Zero Data Loss)
print("\n🔍 Step 5: Staging-to-Fact Row Count Validation:")
val_rows = rs_exec("""
    SELECT 'Orders' AS domain,
           (SELECT COUNT(*) FROM public.stg_orders_raw) AS stg_count,
           (SELECT COUNT(*) FROM public.dw_fact_orders) AS fact_count
    UNION ALL
    SELECT 'Events',
           (SELECT COUNT(*) FROM public.stg_events_raw),
           (SELECT COUNT(*) FROM public.dw_fact_events)
    UNION ALL
    SELECT 'Graph Edges',
           (SELECT COUNT(*) FROM public.stg_edges_raw),
           (SELECT COUNT(*) FROM public.dw_fact_graph_edges);
""", return_results=True)

val_df = pd.DataFrame(val_rows)
val_df["status"] = np.where(val_df["stg_count"] == val_df["fact_count"], "✅ MATCH (0 loss)", "❌ MISMATCH")
display(val_df)

print("\n✅ Task 4 Complete - All staging, dimension, and fact tables loaded and validated in Redshift!")


📊 Step 4: Populating all 3 fact tables...
   ✓ Populated dw_fact_orders
   ✓ Populated dw_fact_events
   ✓ Populated dw_fact_graph_edges

🔍 Step 5: Staging-to-Fact Row Count Validation:


,domain,stg_count,fact_count,status
0,Graph Edges,2500,2500,✅ MATCH (0 loss)
1,Orders,2500,2500,✅ MATCH (0 loss)
2,Events,2500,2500,✅ MATCH (0 loss)



✅ Task 4 Complete - All staging, dimension, and fact tables loaded and validated in Redshift!


---
# Task 5: Optimize Performance and Build OLAP Structures

In this task, you will:
- Verify distribution styles and sort keys are applied
- Run ANALYZE to update statistics
- Create materialized views for common queries

**Deliverables:**
- At least one materialized view for common analytics
- ANALYZE run on key tables

### Task 5 Deliverable: Redshift Distribution & Sort Key Strategy

| Table Name | Table Type | Distribution Strategy | Sort Key | Encoding | Architectural Justification |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `public.dw_fact_orders` | Fact | `DISTKEY(customer_sk)` | `SORTKEY(order_date_key)` | `zstd` | Collocates order rows by customer for fast Customer-360 joins with `dw_fact_events` and `dw_dim_customer`. Sorting by `order_date_key` lets Redshift Zone Maps skip blocks during time-range revenue queries. |
| `public.dw_fact_events` | Fact | `DISTKEY(customer_sk)` | `SORTKEY(event_date_key)` | `zstd` | Matches `dw_fact_orders` distribution on `customer_sk` to eliminate network shuffling during funnel/attribution joins. Sorted by `event_date_key` for fast daily clickstream scans. |
| `public.dw_fact_graph_edges` | Fact | `DISTKEY(to_product_sk)` | `SORTKEY(event_date_key)` | `zstd` | Optimizes product-centric recommendation queries (e.g., `ALSO_BOUGHT_WITH` and product popularity) by collocates edges on the target product. Sorted by `event_date_key` for time-windowed graph analysis. |
| `public.dw_dim_customer` | Dimension | `DISTKEY(customer_id)` | `SORTKEY(customer_id)` | `zstd` | High-cardinality entity dimension (7,057+ rows). Sorting by `customer_id` accelerates SCD Type 2 lookups and merge joins during ETL loads. |
| `public.dw_dim_product` | Dimension | `DISTKEY(product_id)` | `SORTKEY(product_id)` | `zstd` | High-cardinality entity dimension (3,360+ rows). Sorting by `product_id` speeds up product catalog lookups and ETL merges. |
| `public.dw_dim_date` | Dimension | `DISTSTYLE ALL` | `SORTKEY(date_key)` | Default | Small calendar table replicated to every compute node so date joins with all three fact tables happen locally with zero network transfer. |
| **9 Lookup Dimensions** (`dim_channel`, `dim_device`, `dim_browser`, `dim_os`, `dim_referrer`, `dim_campaign`, `dim_payment_method`, `dim_shipping_method`, `dim_ab_variant`) | Dimension | `DISTSTYLE ALL` | None (tiny tables) | `zstd` | Ultra-small reference tables (< 10 rows each). `DISTSTYLE ALL` broadcasts them to every node for instant local hash joins. |

In [23]:
print("=" * 60)
print("TASK 5: Optimize Performance & Build OLAP Structures")
print("=" * 60)

# 1. Create Materialized View for Daily Revenue (DROP + CREATE for idempotency)
print("\n📊 Creating materialized view: public.dw_mv_daily_revenue...")

rs_exec("DROP MATERIALIZED VIEW IF EXISTS public.dw_mv_daily_revenue;")
rs_exec("""
    CREATE MATERIALIZED VIEW public.dw_mv_daily_revenue AS
    SELECT
        order_date_key,
        COUNT(*)             AS orders,
        SUM(order_total_usd) AS revenue_usd,
        AVG(order_total_usd) AS avg_order_value
    FROM public.dw_fact_orders
    GROUP BY order_date_key;
""")
print("   ✓ Materialized view public.dw_mv_daily_revenue created successfully!")

# 2. Execution Proof: Query the materialized view to verify it is populated and fast
mv_count = rs_exec("SELECT COUNT(*) AS mv_days FROM public.dw_mv_daily_revenue;", return_results=True)
print(f"   ✓ Materialized view contains {mv_count[0]['mv_days']:,} pre-aggregated daily summary rows.")

mv_sample = rs_exec("""
    SELECT order_date_key, orders, revenue_usd, avg_order_value
    FROM public.dw_mv_daily_revenue
    ORDER BY order_date_key DESC
    LIMIT 5;
""", return_results=True)

print("\n   Sample rows from public.dw_mv_daily_revenue:")
display(pd.DataFrame(mv_sample))

TASK 5: Optimize Performance & Build OLAP Structures

📊 Creating materialized view: public.dw_mv_daily_revenue...
   ✓ Materialized view public.dw_mv_daily_revenue created successfully!
   ✓ Materialized view contains 542 pre-aggregated daily summary rows.

   Sample rows from public.dw_mv_daily_revenue:


,order_date_key,orders,revenue_usd,avg_order_value
0,20250629,7,1571.26,224.46
1,20250628,5,1207.67,241.53
2,20250627,9,2405.77,267.30
3,20250626,6,2006.26,334.37
4,20250625,5,1913.43,382.68


In [24]:
# 1. Run ANALYZE on key warehouse tables to update Query Optimizer statistics
print("\n📊 Running ANALYZE on key warehouse tables...")

key_tables = [
    'dw_fact_orders',
    'dw_fact_events',
    'dw_fact_graph_edges',
    'dw_dim_customer',
    'dw_dim_product',
    'dw_dim_date'
]

for table in key_tables:
    rs_exec(f"ANALYZE public.{table};")
    print(f"   ✓ ANALYZE complete: public.{table}")

# 2. Verify DISTKEY, SORTKEY, and Compression Encoding directly from Redshift system catalog
print("\n🔍 Verifying Redshift DISTKEY, SORTKEY, and Encoding metadata (pg_table_def):")
opt_meta = rs_exec("""
    SELECT
        tablename,
        "column" AS column_name,
        encoding,
        distkey,
        sortkey
    FROM pg_table_def
    WHERE schemaname = 'public'
      AND tablename LIKE 'dw_%'
      AND (distkey = TRUE OR sortkey != 0)
    ORDER BY tablename, sortkey;
""", return_results=True)

display(pd.DataFrame(opt_meta))

print("\n✅ Task 5 Complete - Performance optimization & OLAP structures verified!")


📊 Running ANALYZE on key warehouse tables...
   ✓ ANALYZE complete: public.dw_fact_orders
   ✓ ANALYZE complete: public.dw_fact_events
   ✓ ANALYZE complete: public.dw_fact_graph_edges
   ✓ ANALYZE complete: public.dw_dim_customer
   ✓ ANALYZE complete: public.dw_dim_product
   ✓ ANALYZE complete: public.dw_dim_date

🔍 Verifying Redshift DISTKEY, SORTKEY, and Encoding metadata (pg_table_def):


,tablename,column_name,encoding,distkey,sortkey
0,dw_dim_customer,customer_id,zstd,True,1
1,dw_dim_date,date_key,none,False,1
2,dw_dim_product,product_id,zstd,True,1
3,dw_fact_events,customer_sk,zstd,True,0
4,dw_fact_events,event_date_key,zstd,False,1
5,dw_fact_graph_edges,to_product_sk,zstd,True,0
6,dw_fact_graph_edges,event_date_key,zstd,False,1
7,dw_fact_orders,customer_sk,zstd,True,0
8,dw_fact_orders,order_date_key,zstd,False,1



✅ Task 5 Complete - Performance optimization & OLAP structures verified!


---
# Task 6: Validate and Report Your Results

In this task, you will:
- Run data quality checks
- Execute sample analytical queries
- Generate the final report

**Deliverables:**
- Data quality checks (row counts, null checks)
- Sample analytical query results
- Final report with schema diagram and design rationale

In [25]:
print("=" * 60)
print("TASK 6: Validation and Reporting")
print("=" * 60)

# 1. Row Counts across ALL Staging, Dimension, Fact, and Materialized View objects
print("\n📊 1. Warehouse Table Row Counts:")
print("-" * 50)

row_counts = rs_exec("""
    SELECT 'Staging'   AS layer, 'stg_orders_raw'         AS table_name, COUNT(*) AS row_count FROM public.stg_orders_raw
    UNION ALL SELECT 'Staging',   'stg_events_raw',         COUNT(*) FROM public.stg_events_raw
    UNION ALL SELECT 'Staging',   'stg_edges_raw',          COUNT(*) FROM public.stg_edges_raw
    UNION ALL SELECT 'Fact',      'dw_fact_orders',         COUNT(*) FROM public.dw_fact_orders
    UNION ALL SELECT 'Fact',      'dw_fact_events',         COUNT(*) FROM public.dw_fact_events
    UNION ALL SELECT 'Fact',      'dw_fact_graph_edges',    COUNT(*) FROM public.dw_fact_graph_edges
    UNION ALL SELECT 'Dimension', 'dw_dim_date',            COUNT(*) FROM public.dw_dim_date
    UNION ALL SELECT 'Dimension', 'dw_dim_customer',        COUNT(*) FROM public.dw_dim_customer
    UNION ALL SELECT 'Dimension', 'dw_dim_product',         COUNT(*) FROM public.dw_dim_product
    UNION ALL SELECT 'Dimension', 'dw_dim_campaign',        COUNT(*) FROM public.dw_dim_campaign
    UNION ALL SELECT 'Dimension', 'dw_dim_channel',         COUNT(*) FROM public.dw_dim_channel
    UNION ALL SELECT 'Dimension', 'dw_dim_device',          COUNT(*) FROM public.dw_dim_device
    UNION ALL SELECT 'Dimension', 'dw_dim_browser',         COUNT(*) FROM public.dw_dim_browser
    UNION ALL SELECT 'Dimension', 'dw_dim_os',              COUNT(*) FROM public.dw_dim_os
    UNION ALL SELECT 'Dimension', 'dw_dim_referrer',        COUNT(*) FROM public.dw_dim_referrer
    UNION ALL SELECT 'Dimension', 'dw_dim_shipping_method', COUNT(*) FROM public.dw_dim_shipping_method
    UNION ALL SELECT 'Dimension', 'dw_dim_payment_method',  COUNT(*) FROM public.dw_dim_payment_method
    UNION ALL SELECT 'Dimension', 'dw_dim_ab_variant',      COUNT(*) FROM public.dw_dim_ab_variant
    UNION ALL SELECT 'OLAP MV',   'dw_mv_daily_revenue',    COUNT(*) FROM public.dw_mv_daily_revenue
    ORDER BY layer, table_name;
""", return_results=True)

row_counts_df = pd.DataFrame(row_counts)
display(row_counts_df)

# 2. Data Quality Checks: Duplicates, NULL Key Checks, and Referential Integrity
print("\n🔍 2. Data Quality & Referential Integrity Checks:")
print("-" * 50)

dq_checks = rs_exec("""
    -- Duplicate Primary Key Checks
    SELECT 'Duplicate order_id in dw_fact_orders' AS check_name,
           COUNT(*) - COUNT(DISTINCT order_id)    AS issue_count
    FROM public.dw_fact_orders
    UNION ALL
    SELECT 'Duplicate event_id in dw_fact_events',
           COUNT(*) - COUNT(DISTINCT event_id)
    FROM public.dw_fact_events
    UNION ALL
    SELECT 'Duplicate edge_id in dw_fact_graph_edges',
           COUNT(*) - COUNT(DISTINCT edge_id)
    FROM public.dw_fact_graph_edges

    -- NULL Foreign Key Checks
    UNION ALL
    SELECT 'NULL customer_sk in dw_fact_orders',
           SUM(CASE WHEN customer_sk IS NULL THEN 1 ELSE 0 END)
    FROM public.dw_fact_orders
    UNION ALL
    SELECT 'NULL order_date_key in dw_fact_orders',
           SUM(CASE WHEN order_date_key IS NULL THEN 1 ELSE 0 END)
    FROM public.dw_fact_orders
    UNION ALL
    SELECT 'NULL customer_sk in dw_fact_events',
           SUM(CASE WHEN customer_sk IS NULL THEN 1 ELSE 0 END)
    FROM public.dw_fact_events

    -- Referential Integrity (Orphan Foreign Key Checks)
    UNION ALL
    SELECT 'Orphan customer_sk in dw_fact_orders (no dim_customer match)',
           COUNT(*)
    FROM public.dw_fact_orders f
    LEFT JOIN public.dw_dim_customer d ON f.customer_sk = d.customer_sk
    WHERE d.customer_sk IS NULL
    UNION ALL
    SELECT 'Orphan order_date_key in dw_fact_orders (no dim_date match)',
           COUNT(*)
    FROM public.dw_fact_orders f
    LEFT JOIN public.dw_dim_date d ON f.order_date_key = d.date_key
    WHERE d.date_key IS NULL;
""", return_results=True)

dq_df = pd.DataFrame(dq_checks)
dq_df["status"] = np.where(dq_df["issue_count"] == 0, "✅ PASS", "⚠️ REVIEW")
display(dq_df)

TASK 6: Validation and Reporting

📊 1. Warehouse Table Row Counts:
--------------------------------------------------


,layer,table_name,row_count
0,Dimension,dw_dim_ab_variant,2
1,Dimension,dw_dim_browser,5
2,Dimension,dw_dim_campaign,7
3,Dimension,dw_dim_channel,5
4,Dimension,dw_dim_customer,7057
5,Dimension,dw_dim_date,552
6,Dimension,dw_dim_device,3
7,Dimension,dw_dim_os,5
8,Dimension,dw_dim_payment_method,5
9,Dimension,dw_dim_product,3360



🔍 2. Data Quality & Referential Integrity Checks:
--------------------------------------------------


,check_name,issue_count,status
0,Duplicate order_id in dw_fact_orders,0,✅ PASS
1,Duplicate event_id in dw_fact_events,0,✅ PASS
2,Orphan customer_sk in dw_fact_orders (no dim_c...,0,✅ PASS
3,Orphan order_date_key in dw_fact_orders (no di...,0,✅ PASS
4,NULL customer_sk in dw_fact_orders,0,✅ PASS
5,Duplicate edge_id in dw_fact_graph_edges,0,✅ PASS
6,NULL order_date_key in dw_fact_orders,0,✅ PASS
7,NULL customer_sk in dw_fact_events,0,✅ PASS


In [26]:
# Helper to measure BOTH wall-clock time and Redshift server-side execution time (ms)
def rs_exec_timed(sql: str):
    """Execute a SELECT query on Redshift and return (rows, wall_sec, server_ms)."""
    t0 = time.time()
    resp = rsd.execute_statement(Sql=sql, **_rs_kwargs())
    stmt_id = resp["Id"]

    while True:
        desc = rsd.describe_statement(Id=stmt_id)
        status = desc["Status"]
        if status == "FINISHED":
            break
        if status in ("FAILED", "ABORTED"):
            raise RuntimeError(f"Redshift SQL {status}: {desc.get('Error')}")
        time.sleep(0.2)

    wall_sec = round(time.time() - t0, 3)
    # Redshift Data API returns Duration in nanoseconds; convert to milliseconds
    server_ms = round(desc.get("Duration", 0) / 1_000_000, 2)

    rows_out = []
    next_token = None
    while True:
        kwargs = {"Id": stmt_id}
        if next_token:
            kwargs["NextToken"] = next_token
        res = rsd.get_statement_result(**kwargs)
        col_names = [c["name"] for c in res["ColumnMetadata"]]
        for record in res["Records"]:
            row_dict = {}
            for col_name, cell in zip(col_names, record):
                if cell.get("isNull"):
                    row_dict[col_name] = None
                elif "stringValue" in cell:
                    row_dict[col_name] = cell["stringValue"]
                elif "longValue" in cell:
                    row_dict[col_name] = cell["longValue"]
                elif "doubleValue" in cell:
                    row_dict[col_name] = cell["doubleValue"]
                elif "booleanValue" in cell:
                    row_dict[col_name] = cell["booleanValue"]
                else:
                    row_dict[col_name] = None
            rows_out.append(row_dict)
        next_token = res.get("NextToken")
        if not next_token:
            break
    return rows_out, wall_sec, server_ms


print("\n📊 3. Performance Optimization Benchmark (Before vs. After) & Analytical Queries:")
print("=" * 70)

# Disable Redshift result caching for this session so timings reflect real execution
rs_exec("SET enable_result_cache_for_session TO OFF;")

# -------------------------------------------------------------------------
# BENCHMARK: Same Daily Revenue Query BEFORE vs. AFTER Materialized View
# -------------------------------------------------------------------------
sql_daily_rev_base = """
SELECT
    d.date_actual,
    d.year,
    d.quarter,
    d.is_weekend,
    COUNT(*)                     AS orders,
    ROUND(SUM(f.order_total_usd), 2) AS revenue_usd,
    ROUND(AVG(f.order_total_usd), 2) AS avg_order_value
FROM public.dw_fact_orders f
JOIN public.dw_dim_date d ON d.date_key = f.order_date_key
GROUP BY d.date_actual, d.year, d.quarter, d.is_weekend
ORDER BY revenue_usd DESC
LIMIT 10;
""".strip()

sql_daily_rev_mv = """
SELECT
    d.date_actual,
    d.year,
    d.quarter,
    d.is_weekend,
    mv.orders,
    ROUND(mv.revenue_usd, 2)     AS revenue_usd,
    ROUND(mv.avg_order_value, 2) AS avg_order_value
FROM public.dw_mv_daily_revenue mv
JOIN public.dw_dim_date d ON d.date_key = mv.order_date_key
ORDER BY mv.revenue_usd DESC
LIMIT 10;
""".strip()

print("\n⚡ BENCHMARK — BEFORE (Direct Aggregation on dw_fact_orders):")
base_rows, base_wall, base_ms = rs_exec_timed(sql_daily_rev_base)
base_rev_df = pd.DataFrame(base_rows)
print(f"   ⏱️ Wall-clock: {base_wall}s | Redshift Server Execution: {base_ms} ms")
display(base_rev_df)

print("\n⚡ BENCHMARK — AFTER (Pre-Aggregated Materialized View dw_mv_daily_revenue):")
mv_rows, mv_wall, mv_ms = rs_exec_timed(sql_daily_rev_mv)
daily_rev_df = pd.DataFrame(mv_rows)
print(f"   ⏱️ Wall-clock: {mv_wall}s | Redshift Server Execution: {mv_ms} ms")
display(daily_rev_df)

# Verify results are 100% equivalent
results_match = base_rev_df.equals(daily_rev_df)
speedup_pct = round(((base_ms - mv_ms) / base_ms) * 100, 1) if base_ms > 0 else 0.0

benchmark_df = pd.DataFrame([
    {
        "Version": "BEFORE: Base Fact Table (dw_fact_orders + GROUP BY)",
        "Rows_Returned": len(base_rev_df),
        "Redshift_Server_ms": base_ms,
        "API_Wall_Clock_sec": base_wall
    },
    {
        "Version": "AFTER: Materialized View (dw_mv_daily_revenue)",
        "Rows_Returned": len(daily_rev_df),
        "Redshift_Server_ms": mv_ms,
        "API_Wall_Clock_sec": mv_wall
    }
])

print("\n📈 Before vs. After Optimization Summary:")
display(benchmark_df)
print(f"   ✅ Equivalent Results Verified: {results_match}")
print(f"   📊 Server-Side Latency Change: {base_ms} ms -> {mv_ms} ms ({speedup_pct}% change)")
print("   💡 Explanation: Both queries return identical Top-10 revenue days. The Materialized View")
print("      eliminates scanning and grouping all 2,500 rows in dw_fact_orders at query time by")
print("      reading 542 pre-aggregated daily rows—a gain that scales dramatically on multi-million-row tables.")

# -------------------------------------------------------------------------
# ANALYTICAL QUERY 2: Channel Revenue, Delivery & Return Rate
# -------------------------------------------------------------------------
sql_channel_perf = """
SELECT
    ch.channel,
    COUNT(*)                                                                 AS total_orders,
    ROUND(SUM(f.order_total_usd), 2)                                         AS total_revenue_usd,
    ROUND(AVG(f.order_total_usd), 2)                                         AS avg_order_value,
    ROUND(AVG(f.delivery_days::DECIMAL(10,2)), 2)                            AS avg_delivery_days,
    ROUND(100.0 * SUM(CASE WHEN f.returned THEN 1 ELSE 0 END) / COUNT(*), 2) AS return_rate_pct
FROM public.dw_fact_orders f
JOIN public.dw_dim_channel ch ON ch.channel_sk = f.channel_sk
GROUP BY ch.channel
ORDER BY total_revenue_usd DESC;
""".strip()

print("\n📦 Query 2: E-Commerce Performance by Sales Channel")
channel_rows, ch_wall, ch_ms = rs_exec_timed(sql_channel_perf)
channel_perf_df = pd.DataFrame(channel_rows)
print(f"   ⏱️ Wall-clock: {ch_wall}s | Redshift Server Execution: {ch_ms} ms")
display(channel_perf_df)

# -------------------------------------------------------------------------
# ANALYTICAL QUERY 3: Neo4j Graph Insights
# -------------------------------------------------------------------------
sql_graph_perf = """
SELECT
    g.relationship,
    COUNT(*)                        AS total_edges,
    COUNT(DISTINCT g.to_product_sk) AS distinct_target_products,
    ROUND(AVG(g.edge_strength), 3)  AS avg_edge_strength,
    ROUND(AVG(g.unit_price_usd), 2) AS avg_unit_price_usd
FROM public.dw_fact_graph_edges g
GROUP BY g.relationship
ORDER BY total_edges DESC;
""".strip()

print("\n🕸️ Query 3: Graph Relationship Activity & Avg Edge Strength (Neo4j -> Redshift)")
graph_rows, gr_wall, gr_ms = rs_exec_timed(sql_graph_perf)
graph_perf_df = pd.DataFrame(graph_rows)
print(f"   ⏱️ Wall-clock: {gr_wall}s | Redshift Server Execution: {gr_ms} ms")
display(graph_perf_df)

# Combined Performance Summary
metrics_df = pd.DataFrame([
    {"Query": "Q1a: Daily Revenue (Base Fact Table - BEFORE)", "Rows_Returned": len(base_rev_df), "Server_Time_ms": base_ms, "Wall_Time_Sec": base_wall},
    {"Query": "Q1b: Daily Revenue (Materialized View - AFTER)", "Rows_Returned": len(daily_rev_df), "Server_Time_ms": mv_ms, "Wall_Time_Sec": mv_wall},
    {"Query": "Q2: Channel Revenue & Return Rate", "Rows_Returned": len(channel_perf_df), "Server_Time_ms": ch_ms, "Wall_Time_Sec": ch_wall},
    {"Query": "Q3: Graph Relationship Analytics", "Rows_Returned": len(graph_perf_df), "Server_Time_ms": gr_ms, "Wall_Time_Sec": gr_wall}
])
print("\n⚡ Complete Query Execution Summary:")
display(metrics_df)


📊 3. Performance Optimization Benchmark (Before vs. After) & Analytical Queries:

⚡ BENCHMARK — BEFORE (Direct Aggregation on dw_fact_orders):
   ⏱️ Wall-clock: 1.278s | Redshift Server Execution: 453.0 ms


,date_actual,year,quarter,is_weekend,orders,revenue_usd,avg_order_value
0,2025-04-26,2025,2,True,9,10005.75,1111.75
1,2024-04-27,2024,2,True,11,8379.88,761.80
2,2025-01-05,2025,1,True,8,7753.02,969.12
3,2024-04-17,2024,2,False,9,7488.27,832.03
4,2025-03-14,2025,1,False,7,7029.01,1004.14
5,2024-01-14,2024,1,True,5,6938.24,1387.64
6,2024-09-16,2024,3,False,2,6434.74,3217.37
7,2024-02-16,2024,1,False,7,6055.87,865.12
8,2024-07-25,2024,3,False,8,6023.05,752.88
9,2024-12-28,2024,4,True,7,5970.72,852.96



⚡ BENCHMARK — AFTER (Pre-Aggregated Materialized View dw_mv_daily_revenue):
   ⏱️ Wall-clock: 0.98s | Redshift Server Execution: 223.51 ms


,date_actual,year,quarter,is_weekend,orders,revenue_usd,avg_order_value
0,2025-04-26,2025,2,True,9,10005.75,1111.75
1,2024-04-27,2024,2,True,11,8379.88,761.80
2,2025-01-05,2025,1,True,8,7753.02,969.12
3,2024-04-17,2024,2,False,9,7488.27,832.03
4,2025-03-14,2025,1,False,7,7029.01,1004.14
5,2024-01-14,2024,1,True,5,6938.24,1387.64
6,2024-09-16,2024,3,False,2,6434.74,3217.37
7,2024-02-16,2024,1,False,7,6055.87,865.12
8,2024-07-25,2024,3,False,8,6023.05,752.88
9,2024-12-28,2024,4,True,7,5970.72,852.96



📈 Before vs. After Optimization Summary:


,Version,Rows_Returned,Redshift_Server_ms,API_Wall_Clock_sec
0,BEFORE: Base Fact Table (dw_fact_orders + GROU...,10,453.00,1.278
1,AFTER: Materialized View (dw_mv_daily_revenue),10,223.51,0.980


... (2 outputs left out)


🕸️ Query 3: Graph Relationship Activity & Avg Edge Strength (Neo4j -> Redshift)
   ⏱️ Wall-clock: 1.304s | Redshift Server Execution: 457.52 ms


,relationship,total_edges,distinct_target_products,avg_edge_strength,avg_unit_price_usd
0,VIEWED,830,789,0.555,138.60
1,PURCHASED,469,458,0.551,140.36
2,ADDED_TO_CART,441,430,0.533,149.58
3,ALSO_BOUGHT_WITH,393,382,0.546,143.13
4,REFERRED_FRIEND,256,0,0.532,138.18
5,RETURNS,111,110,0.555,165.03



⚡ Complete Query Execution Summary:


,Query,Rows_Returned,Server_Time_ms,Wall_Time_Sec
0,Q1a: Daily Revenue (Base Fact Table - BEFORE),10,453.00,1.278
1,Q1b: Daily Revenue (Materialized View - AFTER),10,223.51,0.980
2,Q2: Channel Revenue & Return Rate,5,382.45,1.273
3,Q3: Graph Relationship Analytics,6,457.52,1.304


In [27]:
print("\n📄 Generating Final Report...")

def df_to_md(df: pd.DataFrame) -> str:
    """Convert a Pandas DataFrame to a clean Markdown table without external dependencies."""
    headers = list(df.columns)
    lines = [
        "| " + " | ".join(str(h) for h in headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |"
    ]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(row[h]) for h in headers) + " |")
    return "\n".join(lines)

# Load Mermaid ERD from project-mermaid-diagram.md
with open(MERMAID_MD, "r", encoding="utf-8") as f:
    mermaid_raw = f.read().strip()

mermaid_lines = [ln for ln in mermaid_raw.splitlines() if not ln.strip().startswith("#")]
mermaid_clean = "\n".join(mermaid_lines).strip()

fence = "`" * 3

report_content = f"""# E-Commerce Data Warehouse Build Report

**Generated:** {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC  
**Target Platform:** Amazon Redshift (`{REDSHIFT_DATABASE}` / Workgroup `{REDSHIFT_WORKGROUP}`)  
**Integrated Sources:** PostgreSQL (Orders), Apache Cassandra (Clickstream Events), Neo4j (Graph Relationships)

---

## 1. Executive Summary
This project unifies three heterogeneous operational databases—**PostgreSQL**, **Apache Cassandra**, and **Neo4j**—into a centralized, Kimball-style **Star Schema** in **Amazon Redshift**. The pipeline enforces strict data hygiene (standardizing ISO country codes `'UK'` -> `'GB'`, resolving polymorphic graph nodes into explicit role-playing foreign keys, and cleaning null campaign values) while preserving **100% row completeness** (2,500 rows per source) from staging to final fact tables.

---

## 2. Warehouse Schema Diagram (Mermaid.js ERD)

{fence}mermaid
{mermaid_clean}
{fence}

---

## 3. Table Inventory & Row Counts

{df_to_md(row_counts_df)}

---

## 4. Data Quality & Referential Integrity Validation

{df_to_md(dq_df)}

---

## 5. Design Rationale & Redshift Optimization Strategy

1. **Why Star Schema (Kimball Methodology)?**  
   Separating quantitative measures into three grain-pure fact tables (`dw_fact_orders`, `dw_fact_events`, `dw_fact_graph_edges`) and descriptive attributes into 12 conformed dimensions (`dw_dim_customer`, `dw_dim_product`, `dw_dim_date`, and 9 lookup dimensions) simplifies BI joins and guarantees consistent KPIs across teams.
2. **Distribution Strategy (`DISTKEY` vs. `DISTSTYLE ALL`):**  
   - `dw_fact_orders` and `dw_fact_events` share `DISTKEY(customer_sk)` to collocate customer records on the same compute node, eliminating network shuffling during Customer-360 and funnel joins.
   - `dw_fact_graph_edges` uses `DISTKEY(to_product_sk)` to optimize product co-purchase (`ALSO_BOUGHT_WITH`) and recommendation queries.
   - `dw_dim_date` and all 9 small lookup dimensions use `DISTSTYLE ALL` for instant local broadcast joins.
3. **Sort Keys & Compression:**  
   All fact tables use chronological `SORTKEY`s (`order_date_key`, `event_date_key`) so Redshift Zone Maps skip irrelevant disk blocks during date-range filtering. `ENCODE zstd` minimizes storage footprint and I/O.
4. **OLAP Materialized View & Statistics:**  
   `public.dw_mv_daily_revenue` pre-aggregates daily order counts, revenue, and AOV, and `ANALYZE` was executed across all core fact and dimension tables to keep the Redshift Query Optimizer accurate.

---

## 6. Performance Optimization Benchmark (Before vs. After)

To demonstrate the impact of the `dw_mv_daily_revenue` Materialized View, we disabled session result caching (`SET enable_result_cache_for_session TO OFF`) and ran the exact same Top-10 Daily Revenue aggregation directly against `dw_fact_orders` (Before) and against `dw_mv_daily_revenue` (After).

### Benchmark Results
{df_to_md(benchmark_df)}

* **Equivalent Output Verified:** `{results_match}` (both queries return identical Top-10 revenue days).
* **Analysis:** The Materialized View avoids scanning and grouping all 2,500 base fact rows on every request by reading 542 pre-aggregated daily rows. While Redshift Data API polling adds fixed network overhead (~1s wall-clock), Redshift's server-side execution time dropped from **{base_ms} ms** to **{mv_ms} ms**, and this pre-aggregation advantage scales linearly as order volume grows into millions of rows.

### Before SQL (Direct Fact-to-Dimension Aggregation)
{fence}sql
{sql_daily_rev_base}
{fence}

### After SQL (Optimized Materialized View Query)
{fence}sql
{sql_daily_rev_mv}
{fence}

### Returned Top-10 Daily Revenue Sample
{df_to_md(daily_rev_df)}

---

## 7. Example Analytical Queries & Results

### Example 1: E-Commerce Performance by Sales Channel (`dw_fact_orders` JOIN `dw_dim_channel`)
This query joins the Orders fact table to the broadcast-distributed Channel dimension (`DISTSTYLE ALL`) to compare revenue, average order value, fulfillment speed, and return rates across sales channels.

{fence}sql
{sql_channel_perf}
{fence}

**Query Result:**
{df_to_md(channel_perf_df)}

**Insight:** Mobile apps (`android_app` and `ios_app`) drive the highest total revenue ($224K and $221K) and maintain the lowest return rates (13.86%–14.58%), whereas `web` shows a noticeably higher return rate (20.40%) despite similar delivery times (~4 days).

---

### Example 2: Graph Relationship Activity (`dw_fact_graph_edges`)
This query summarizes flattened Neo4j graph edges stored in Redshift to evaluate customer-to-product and product-to-product engagement across relationship types.

{fence}sql
{sql_graph_perf}
{fence}

**Query Result:**
{df_to_md(graph_perf_df)}

**Insight:** `VIEWED` (830 edges) and `PURCHASED` (469 edges) dominate graph activity, while `RETURNS` edges correlate with the highest average unit price ($165.03), highlighting higher-ticket items as a key focus area for return reduction.

---

## 8. All Query Execution Metrics
{df_to_md(metrics_df)}

---

## 9. Analytics Capabilities Supported
- **Executive Revenue & Margin Tracking:** Instant daily/quarterly revenue, discount impact, and AOV reporting via `dw_mv_daily_revenue` and `dw_dim_date`.
- **Omnichannel & Fulfillment Analytics:** Compare conversion, delivery speed, and return rates across web, mobile apps, and marketplaces.
- **Clickstream Funnel & Fraud Monitoring:** Analyze session depth, latency, A/B test variants, and fraud scores alongside completed orders.
- **Graph-Driven Personalization:** Measure `ALSO_BOUGHT_WITH`, `REFERRED_FRIEND`, and `VIEWED` product affinity directly inside SQL BI tools.
"""

# Save report to disk
report_path = os.path.join(BASE_DIR, "warehouse_report.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_content)

print(f"\n✅ Final executive report saved to: {report_path}")
print("\n" + "=" * 60)
print("ALL 6 TASKS COMPLETED SUCCESSFULLY! ✅")
print("=" * 60)


📄 Generating Final Report...

✅ Final executive report saved to: ./warehouse_report.md

ALL 6 TASKS COMPLETED SUCCESSFULLY! ✅
